# 06 — Corrección de Anomalías

Aplicamos estrategias de corrección específicas para cada tipo de anomalía detectada:
1. **Datos Faltantes** → IterativeImputer con RandomForestRegressor (multivariable)
2. **Sensor Atascado** → Método híbrido: marcar como NaN + IterativeImputer
3. **Ruido** → Interpolación local (media de vecino anterior y siguiente)
4. **Fuera de Rango** → Interpolación local con umbrales adaptativos (v3 M4)
5. **Desviación de Correlación** → Interpolación local con umbrales adaptativos
6. **Contextual** → Interpolación local con umbrales adaptativos

**Entrada:** `data/interim/02_datos_inyectados.parquet` + modelos  
**Salida:** `data/interim/06_datos_corregidos.parquet`

In [ ]:
from collections import defaultdict
# ─── Intel Extension for Scikit-learn ────────────────────────────────────────
# IMPORTANTE: debe cargarse ANTES que cualquier import de sklearn.
# Instalar con: pip install scikit-learn-intelex
# En CPU Intel puede acelerar Random Forest e IterativeImputer 10-100x.
try:
    from sklearnex import patch_sklearn
    patch_sklearn()
    _intel_activo = True
except ImportError:
    _intel_activo = False

# ─── Librerías estándar ───────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import os, glob, joblib, pickle, warnings
import matplotlib.ticker as mticker

# ─── Scikit-learn (ya parcheado si Intel Extension está disponible) ───────────
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.experimental import enable_iterative_imputer  # requerido antes de IterativeImputer
from sklearn.impute import SimpleImputer, IterativeImputer
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                              classification_report, confusion_matrix,
                              average_precision_score,
                              mean_squared_error, mean_absolute_error)
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 100

# ─── Configuración del proyecto ───────────────────────────────────────────────
import sys
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
from config import *

if _intel_activo:
    print("Intel Extension for Scikit-learn ACTIVA — algoritmos acelerados con oneAPI")
else:
    print("Tip: instala scikit-learn-intelex para acelerar en CPU Intel")
    print("     pip install scikit-learn-intelex")


## 0. Cargar datos y modelos

In [ ]:
# Cargamos el dataset con anomalías inyectadas (antes de las features para tener solo columnas originales)
df_trabajo_orig = pd.read_parquet(PARQUET_02)

# Cargamos el dataset con features para hacer las predicciones de los modelos
df_con_features = pd.read_parquet(PARQUET_04)

# Cargamos los modelos entrenados
rf_detector         = joblib.load(MODELO_1_PATH)
imputer             = joblib.load(IMPUTER_M1_PATH)
features_esperadas  = joblib.load(FEATURES_M1_PATH)
rf_clasificador_tipo = joblib.load(MODELO_2_PATH)
label_encoder_model2 = joblib.load(LABEL_ENC_M2_PATH)

print("Modelos cargados correctamente.")
print(f"Dataset original: {df_trabajo_orig.shape}")

# ─── Cargar datos originales limpios (antes de inyección) ────────────────────
df_original = pd.read_parquet(PARQUET_01)
print(f"Datos originales cargados: {df_original.shape}")

# ─── Generar pred_tipo_anomalia para todo el dataset ─────────────────────────
excluir_inf = COLUMNAS_EXCLUIR_FEATURES
columnas_features_inf = [c for c in df_con_features.columns if c not in excluir_inf]
X_todo = df_con_features[columnas_features_inf].select_dtypes(include='number').copy()

cols_orig_feats = [c for c in features_esperadas if not c.startswith('missingindicator_')]
cols_disp_feats = [c for c in cols_orig_feats if c in X_todo.columns]
X_todo_imp_np = imputer.transform(X_todo[cols_disp_feats])
X_todo_imp = pd.DataFrame(
    X_todo_imp_np,
    columns=imputer.get_feature_names_out(cols_disp_feats),
    index=X_todo.index)

pred_det_todo = rf_detector.predict(X_todo_imp)
df_trabajo_orig['pred_tipo_anomalia'] = pd.NA
mask_anom = pred_det_todo == 'anomalia'
if mask_anom.any():
    X_anom = X_todo_imp[mask_anom]
    pred_tipo_enc = rf_clasificador_tipo.predict(X_anom)
    pred_tipo_str = label_encoder_model2.inverse_transform(pred_tipo_enc)
    df_trabajo_orig.loc[X_todo.index[mask_anom], 'pred_tipo_anomalia'] = pred_tipo_str

print(f"pred_tipo_anomalia añadida. Anomalías detectadas: {mask_anom.sum()}")
print(df_trabajo_orig['pred_tipo_anomalia'].value_counts(dropna=False).head(10))


## 1. Definir columnas objetivo y preparar DataFrame

In [ ]:
# df_trabajo_orig contiene las columnas del dataset con anomalías inyectadas:
columnas_sensores_y_actuadores = [
    'PCO2EXT', 'PHEXT', 'PRAD', 'PRGINT', 'PTEXT', 'PVV',
    'XCO2I', 'XHINV', 'XTINV', 'XTS'
]

# Esta es la variable que se usará más adelante
columnas_objetivo_existentes = [col for col in columnas_sensores_y_actuadores if col in df_trabajo_orig.columns]

# Definimos df_para_corregir como una copia del DataFrame base
df_para_corregir = df_trabajo_orig.copy()

print("columnas_objetivo_existentes:", columnas_objetivo_existentes)
print("df_para_corregir.shape:", df_para_corregir.shape)

## 2. Preparar IterativeImputer para Datos Faltantes
El IterativeImputer usa RandomForestRegressor para predecir el valor de cada NaN a partir del resto de columnas. Es lento en datasets grandes pero preciso.

In [ ]:
if 'df_para_corregir' in locals() and df_para_corregir is not None and 'columnas_objetivo_existentes' in locals():
    
    # Vamos a redefinir columnas_para_imputacion basándonos en df_para_corregir
    # Excluiremos las columnas de etiquetas/predicciones que añadimos
    columnas_a_excluir_de_imputacion = ['Fecha', 'etiqueta_deteccion', 'etiqueta_tipo_anomalia', 'pred_tipo_anomalia']
    columnas_potenciales_para_imputacion = [col for col in df_para_corregir.columns if col not in columnas_a_excluir_de_imputacion]
    
    columnas_numericas_para_imputacion = df_para_corregir[columnas_potenciales_para_imputacion].select_dtypes(include=np.number).columns.tolist()

    if not columnas_numericas_para_imputacion:
        print("Advertencia: No se encontraron columnas numéricas adecuadas para la imputación iterativa en 'df_para_corregir'.")
        iterative_imputer_faltantes = None # Usar un nombre específico para este imputador
    else:
        print(f"Columnas que se considerarán para IterativeImputer (de df_para_corregir): {columnas_numericas_para_imputacion}")

        # Crear una copia de df_para_corregir para realizar la corrección
        df_corregido_faltantes = df_para_corregir.copy() # Usar un nombre específico

        iterative_imputer_faltantes = IterativeImputer(
            estimator=RandomForestRegressor(**IMPUTER_RF_PARAMS),
            max_iter=IMPUTER_MAX_ITER,
            random_state=IMPUTER_RF_PARAMS['random_state'],
            initial_strategy='median',
            imputation_order='ascending',
            verbose=1
        )
else:
    print("Error: 'df_para_corregir' o 'columnas_objetivo_existentes' no están definidos o df_para_corregir no se cargó.")
    iterative_imputer_faltantes = None
    df_corregido_faltantes = None # Asegurar que no esté definido si hay error
    columnas_numericas_para_imputacion = []

## 3. Aplicar IterativeImputer — Datos Faltantes

In [ ]:
if iterative_imputer_faltantes is not None and df_corregido_faltantes is not None and columnas_numericas_para_imputacion:

    subset_indices_faltantes = df_corregido_faltantes.index # Índice del DataFrame actual

    print(f"\nValores NaN en 'df_corregido_faltantes' ANTES de IterativeImputer (para columnas seleccionadas):")
    nans_antes_faltantes = df_corregido_faltantes[columnas_numericas_para_imputacion].isnull().sum()
    print(nans_antes_faltantes[nans_antes_faltantes > 0])
    total_nans_antes_faltantes = nans_antes_faltantes.sum()

    print("\nEjecutando IterativeImputer para 'Datos Faltantes' (esto puede tardar un poco)...")
    # Aplicar fit_transform al subset de df_corregido_faltantes que tiene las columnas numéricas
    df_subset_imputado_np_faltantes = iterative_imputer_faltantes.fit_transform(df_corregido_faltantes[columnas_numericas_para_imputacion])

    df_subset_imputado_faltantes = pd.DataFrame(df_subset_imputado_np_faltantes, columns=columnas_numericas_para_imputacion, index=subset_indices_faltantes)

    if 'pred_tipo_anomalia' in df_para_corregir.columns:
        mask_pred_faltantes = df_para_corregir['pred_tipo_anomalia'] == 'Datos Faltantes'
    else:
        mask_pred_faltantes = pd.Series(True, index=df_para_corregir.index) # Todo en True para depender solo del NaN

    for col in columnas_numericas_para_imputacion:
        # Crear una máscara de los valores que son NaN originalmente en esta columna
        mask_nan = df_para_corregir[col].isnull()

        # Combinamos ambas máscaras para obtener solo las celdas exactas que debemos reemplazar
        mask_a_corregir = mask_pred_faltantes & mask_nan
        
        # Reemplazamos solo las anomalías clasificadas como "Datos Faltantes"
        df_corregido_faltantes.loc[mask_a_corregir, col] = df_subset_imputado_faltantes.loc[mask_a_corregir, col]

    print("\nIterativeImputer para 'Datos Faltantes' completado.")
    nans_despues_faltantes = df_corregido_faltantes[columnas_numericas_para_imputacion].isnull().sum().sum()
    print(f"Total NaNs antes: {total_nans_antes_faltantes}, Total NaNs después: {nans_despues_faltantes}")
    print(f"Se corrigieron {total_nans_antes_faltantes - nans_despues_faltantes} anomalías de 'Datos Faltantes'.")

    if 'df_original' in locals(): # Necesita el df_original original, limpio.
        print("\n--- Evaluación de la Calidad de Imputación de Datos Faltantes (comparando con df_original) ---")
        all_true_values_faltantes_eval = []
        all_imputed_values_faltantes_eval = []
        column_metrics_faltantes_eval = []

        for col in columnas_numericas_para_imputacion:
            
            # Obtener los índices de df_para_corregir (que es el test set)
            indices_test_set = df_para_corregir.index
            
            # Identificar dónde df_trabajo_orig tenía NaNs dentro de este test set
            nan_mask_en_trabajo_para_test_set = df_trabajo_orig.loc[indices_test_set, col].isnull()
            
            # Asegurarnos de que estamos evaluando únicamente los que el modelo predijo como "Datos Faltantes"
            if 'pred_tipo_anomalia' in df_para_corregir.columns:
                mask_eval_pred = df_para_corregir.loc[indices_test_set, 'pred_tipo_anomalia'] == 'Datos Faltantes'
            else:
                mask_eval_pred = pd.Series(True, index=indices_test_set)
            
            mask_evaluacion_final = nan_mask_en_trabajo_para_test_set & mask_eval_pred
            indices_nan_evaluables = indices_test_set[mask_evaluacion_final]

            if len(indices_nan_evaluables) == 0:
                continue

            true_values_eval = df_original.loc[indices_nan_evaluables, col]
            imputed_values_eval = df_corregido_faltantes.loc[indices_nan_evaluables, col]
            
            valid_comparison_mask_eval = true_values_eval.notna()
            true_values_valid_eval = true_values_eval[valid_comparison_mask_eval]
            imputed_values_valid_eval = imputed_values_eval[valid_comparison_mask_eval]

            if len(true_values_valid_eval) == 0:
                continue
            
            all_true_values_faltantes_eval.extend(true_values_valid_eval.tolist())
            all_imputed_values_faltantes_eval.extend(imputed_values_valid_eval.tolist())
            
            rmse_eval = np.sqrt(mean_squared_error(true_values_valid_eval, imputed_values_valid_eval))
            mae_eval = mean_absolute_error(true_values_valid_eval, imputed_values_valid_eval)
            
            column_metrics_faltantes_eval.append({
                'Columna': col, 'Num_Val_Evaluados': len(true_values_valid_eval),
                'RMSE': rmse_eval, 'MAE': mae_eval,
                'Media_Real': true_values_valid_eval.mean(), 'Media_Imputada': imputed_values_valid_eval.mean()
            })

        if column_metrics_faltantes_eval:
            df_imputation_metrics_faltantes_eval = pd.DataFrame(column_metrics_faltantes_eval)
            print("\nMétricas de Calidad de Imputación (Datos Faltantes en df_resultados_test_con_predicciones):")
            print(df_imputation_metrics_faltantes_eval.to_string())

            if all_true_values_faltantes_eval and all_imputed_values_faltantes_eval:
                overall_rmse_faltantes_eval = np.sqrt(mean_squared_error(all_true_values_faltantes_eval, all_imputed_values_faltantes_eval))
                overall_mae_faltantes_eval = mean_absolute_error(all_true_values_faltantes_eval, all_imputed_values_faltantes_eval)
                print(f"\nRMSE Global de Imputación (Datos Faltantes): {overall_rmse_faltantes_eval:.3f}")
                print(f"MAE Global de Imputación (Datos Faltantes): {overall_mae_faltantes_eval:.3f}")
        else:
            print("\nNo se evaluaron valores imputados para 'Datos Faltantes' (quizás no había NaNs predichos como tal en df_para_corregir o en df_original para comparar).")

else:
    print("No se pudo ejecutar la corrección de 'Datos Faltantes' porque las variables necesarias no están definidas o no hay columnas para imputar.")

## 4. Evaluar calidad de imputación de Datos Faltantes
Comparamos el valor original con el imputado donde teníamos NaN inyectado artificialmente (tenemos ground truth).

In [ ]:
if ('df_original' in locals() and 
    'df_trabajo_orig' in locals() and 
    'df_corregido_faltantes' in locals() and
    'columnas_numericas_para_imputacion' in locals() and
    columnas_numericas_para_imputacion):

    print("--- Evaluación de la Calidad de Imputación de IterativeImputer ---")
    
    all_true_values = []
    all_imputed_values = []
    column_metrics = []

    # Obtener el índice del DataFrame que fue realmente imputado
    # (df_corregido_faltantes, que se basa en df_para_corregir/X_test)
    index_of_imputed_subset = df_corregido_faltantes.index

    for col in columnas_numericas_para_imputacion:
        # 1. Identificar todos los índices en el df_trabajo_orig completo donde 'col' era NaN
        all_nan_indices_in_df_trabajo_orig = df_trabajo_orig[df_trabajo_orig[col].isnull()].index
        
        if all_nan_indices_in_df_trabajo_orig.empty:
            # print(f"No se encontraron NaNs originales en la columna '{col}' de df_trabajo_orig para evaluar.")
            continue

        # 2. Filtrar estos índices: considerar solo aquellos que también existen en el DataFrame imputado (df_corregido_faltantes)
        #    Esto es crucial porque df_corregido_faltantes es un subconjunto (el conjunto de prueba).
        indices_a_evaluar = index_of_imputed_subset.intersection(all_nan_indices_in_df_trabajo_orig)

        # Filtrar adicionalmente para evaluar solo lo que el modelo identificó como "Datos Faltantes"
        if 'pred_tipo_anomalia' in df_para_corregir.columns:
            mask_pred_eval_global = df_para_corregir.loc[indices_a_evaluar, 'pred_tipo_anomalia'] == 'Datos Faltantes'
            indices_a_evaluar = indices_a_evaluar[mask_pred_eval_global]
        
        if indices_a_evaluar.empty:
            # print(f"No hay NaNs de df_trabajo_orig para la columna '{col}' que correspondan a filas en df_corregido_faltantes para evaluar.")
            continue

        # Obtener los valores verdaderos de df_original para estas celdas específicas
        true_values = df_original.loc[indices_a_evaluar, col]
        
        # Obtener los valores imputados de df_corregido_faltantes para esas mismas celdas
        # Esta línea ahora debería funcionar sin KeyError
        imputed_values = df_corregido_faltantes.loc[indices_a_evaluar, col]
        
        # Es importante asegurarse de que estamos comparando solo donde true_values no sea NaN
        valid_comparison_mask = true_values.notna()
        true_values_valid = true_values[valid_comparison_mask]
        imputed_values_valid = imputed_values[valid_comparison_mask]

        if len(true_values_valid) == 0:
            # print(f"No hay valores verdaderos válidos en df_original para la columna '{col}' en los índices de NaN evaluados. No se puede evaluar.")
            continue
            
        all_true_values.extend(true_values_valid.tolist())
        all_imputed_values.extend(imputed_values_valid.tolist())
        
        # Calcular métricas para esta columna
        rmse = np.sqrt(mean_squared_error(true_values_valid, imputed_values_valid))
        mae = mean_absolute_error(true_values_valid, imputed_values_valid)
        
        column_metrics.append({
            'Columna': col,
            'Num_Valores_Imputados_Evaluados': len(true_values_valid),
            'RMSE': rmse,
            'MAE': mae,
            'Media_Real': true_values_valid.mean(),
            'Media_Imputada': imputed_values_valid.mean()
        })
        # print(f"Columna: {col}, RMSE: {rmse:.3f}, MAE: {mae:.3f}, N_eval: {len(true_values_valid)}")

    if column_metrics:
        df_imputation_metrics = pd.DataFrame(column_metrics)
        print("\nMétricas de Calidad de Imputación por Columna (comparando con df_original):")
        print(df_imputation_metrics.to_string())

        # Calcular métricas globales si hay valores para comparar
        if all_true_values and all_imputed_values:
            overall_rmse = np.sqrt(mean_squared_error(all_true_values, all_imputed_values))
            overall_mae = mean_absolute_error(all_true_values, all_imputed_values)
            print(f"\nRMSE Global de Imputación: {overall_rmse:.3f}")
            print(f"MAE Global de Imputación: {overall_mae:.3f}")

            # Gráfico de Dispersión: Valores Reales vs. Imputados
            plt.figure(figsize=(8, 8))
            sample_size = min(len(all_true_values), 2000) # Tomar una muestra
            if sample_size > 0: # Solo tomar muestra si hay puntos
                sample_indices = np.random.choice(len(all_true_values), sample_size, replace=False)
            
                plt.scatter(
                    np.array(all_true_values)[sample_indices], 
                    np.array(all_imputed_values)[sample_indices], 
                    alpha=0.5, s=10, label='Corrected Points', c='blue'
                )
                min_val = min(min(all_true_values), min(all_imputed_values))
                max_val = max(max(all_true_values), max(all_imputed_values))
                plt.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Ideal')
                plt.xlabel("True Values")
                plt.ylabel("Corrected Values")
                plt.legend()
                plt.grid(True)
                plt.tight_layout()
                plt.show()
            else:
                print("\nNo hay puntos suficientes para el gráfico de dispersión global.")
        else:
            print("\nNo hay suficientes valores válidos comparables para calcular métricas globales o graficar.")
            
    else:
        print("\nNo se pudieron calcular métricas de imputación para ninguna columna (quizás no había NaNs relevantes o valores verdaderos para comparar).")
else:
    print("Error: DataFrames necesarios ('df_original', 'df_trabajo_orig', 'df_corregido_faltantes') o 'columnas_numericas_para_imputacion' no están definidos o la lista de columnas está vacía.")

## 5. Corrección de Sensor Atascado
Estrategia híbrida: primero marcamos los segmentos atascados como NaN, luego aplicamos el mismo IterativeImputer.

In [ ]:
# --- Función para marcar NaNs ---
def marcar_segmentos_stuck_como_nan_condicional(df_a_marcar, columna_nombre,
                                                predicciones_tipo_M2_series,
                                                nombre_clase_sensor_atascado="Sensor Atascado",
                                                min_duracion_atasco=3):
    valores = df_a_marcar[columna_nombre].values
    indices_df = df_a_marcar.index
    n = len(valores)
    segmentos_nanificados_en_col = []
    nan_count_col = 0
    i = 0
    while i < n:
        if pd.isna(valores[i]):
            i += 1
            continue
        j = i
        while j < n and pd.notna(valores[j]) and valores[j] == valores[i]:
            j += 1
        duracion_actual_atasco = j - i
        fue_clasificado_como_atasco = False
        if duracion_actual_atasco >= min_duracion_atasco:
            indices_reales_segmento_df = indices_df[i:j]
            if not indices_reales_segmento_df.empty:
                # Asegurar compatibilidad de índices antes de la intersección
                if not predicciones_tipo_M2_series.empty and isinstance(predicciones_tipo_M2_series.index, type(indices_df)):
                    indices_validos_en_predicciones = predicciones_tipo_M2_series.index.intersection(indices_reales_segmento_df)
                    if not indices_validos_en_predicciones.empty:
                        if (predicciones_tipo_M2_series.loc[indices_validos_en_predicciones] == nombre_clase_sensor_atascado).any():
                            fue_clasificado_como_atasco = True
                elif predicciones_tipo_M2_series.empty: # Si no hay predicciones, no se puede clasificar como atasco
                    pass # fue_clasificado_como_atasco remains False
                else: # Fallback o advertencia si los tipos de índice no coinciden y las predicciones no están vacías
                    print(f"Advertencia (marcar_segmentos): Incompatibilidad de tipos de índice entre df y predicciones para la columna '{columna_nombre}'. Segmento {indices_df[i] if i < len(indices_df) else 'END'} no se considerará atasco.")


        if fue_clasificado_como_atasco:
            # Esta sección se ejecuta solo si fue_clasificado_como_atasco es True
            idx_inicio_segmento_df = indices_reales_segmento_df[0]
            idx_fin_segmento_df = indices_reales_segmento_df[-1]
            if columna_nombre in df_a_marcar.columns:
                df_a_marcar.loc[idx_inicio_segmento_df:idx_fin_segmento_df, columna_nombre] = np.nan
                segmentos_nanificados_en_col.append((idx_inicio_segmento_df, idx_fin_segmento_df))
                nan_count_col += 1
            else:
                print(f"Advertencia (marcar_segmentos): Columna '{columna_nombre}' no encontrada. No se marcará NaN.")
        
        if fue_clasificado_como_atasco and duracion_actual_atasco >= min_duracion_atasco :
            i = j
        elif duracion_actual_atasco > 0 : # Avanzar incluso si no fue clasificado o no cumplió min_duracion_atasco pero hubo segmento
            i = j
        else: # Caso de valor aislado o fin de datos no procesados como segmento
            i += 1
            
    return nan_count_col, segmentos_nanificados_en_col

# --- Función para corregir con interpolación y evaluar---
def corregir_sensor_atascado_condicional_eval(df_a_corregir, columna_nombre,
                                                predicciones_tipo_M2_series,
                                                nombre_clase_sensor_atascado="Sensor Atascado",
                                                min_duracion_atasco=3):
    valores = df_a_corregir[columna_nombre].values
    indices_df = df_a_corregir.index
    n = len(valores)
    segmentos_corregidos_count = 0
    segmentos_marcados_para_interpolacion = []
    i = 0
    while i < n:
        if pd.isna(valores[i]):
            i += 1
            continue
        j = i
        while j < n and pd.notna(valores[j]) and valores[j] == valores[i]:
            j += 1
        duracion_actual_atasco = j - i
        fue_clasificado_como_atasco = False
        if duracion_actual_atasco >= min_duracion_atasco:
            indices_reales_segmento_df = indices_df[i:j]
            if not indices_reales_segmento_df.empty:
                # Asegurar compatibilidad de índices antes de la intersección
                if not predicciones_tipo_M2_series.empty and isinstance(predicciones_tipo_M2_series.index, type(indices_df)):
                    indices_validos_en_predicciones = predicciones_tipo_M2_series.index.intersection(indices_reales_segmento_df)
                    if not indices_validos_en_predicciones.empty:
                        if (predicciones_tipo_M2_series.loc[indices_validos_en_predicciones] == nombre_clase_sensor_atascado).any():
                            fue_clasificado_como_atasco = True
                elif predicciones_tipo_M2_series.empty:
                    pass
                else:
                    print(f"Advertencia (corregir_sensor_atascado): Incompatibilidad de tipos de índice para la columna '{columna_nombre}'. Segmento {indices_df[i] if i < len(indices_df) else 'END'} no se considerará atasco.")


        if fue_clasificado_como_atasco:
            idx_inicio_segmento_df = indices_reales_segmento_df[0]
            idx_fin_segmento_df = indices_reales_segmento_df[-1]
            if columna_nombre in df_a_corregir.columns:
                df_a_corregir.loc[idx_inicio_segmento_df:idx_fin_segmento_df, columna_nombre] = np.nan
                segmentos_marcados_para_interpolacion.append((idx_inicio_segmento_df, idx_fin_segmento_df))
                segmentos_corregidos_count += 1
            else:
                print(f"Advertencia (corregir_sensor_atascado): Columna '{columna_nombre}' no encontrada. No se marcará NaN.")
        
        if fue_clasificado_como_atasco and duracion_actual_atasco >= min_duracion_atasco:
            i = j
        elif duracion_actual_atasco > 0:
            i = j
        else:
            i += 1
            
    if segmentos_corregidos_count > 0:
        if columna_nombre in df_a_corregir.columns:
            df_a_corregir[columna_nombre] = df_a_corregir[columna_nombre].interpolate(method='linear', limit_direction='both')
        else:
            print(f"Advertencia (interpolación): Columna '{columna_nombre}' no encontrada para interpolar.")
    return segmentos_corregidos_count, segmentos_marcados_para_interpolacion

# --- Función de correción dinamicos ---
def corregir_dinamicos_con_estacional_simple(
    df_a_corregir,
    df_historico_referencia,
    columna_a_corregir,
    start_idx_anomalia,
    end_idx_anomalia,
    columna_hora,
    ventana_dias_previos=7,
    metodo_agregacion='median',
    min_valores_historicos=1
):
    if not isinstance(df_a_corregir.index, pd.DatetimeIndex):
        print(f" ALERTA CRÍTICA en 'corregir_dinamicos_con_estacional_simple': df_a_corregir (columna '{columna_a_corregir}') NO tiene DatetimeIndex. Saltando.")
        return
    if not isinstance(df_historico_referencia.index, pd.DatetimeIndex):
        print(f" ALERTA CRÍTICA en 'corregir_dinamicos_con_estacional_simple': df_historico_referencia (para '{columna_a_corregir}') NO tiene DatetimeIndex. Saltando.")
        return
    if columna_hora not in df_a_corregir.columns:
        print(f" Error en 'corregir_dinamicos_con_estacional_simple': Columna '{columna_hora}' no encontrada en df_a_corregir. Saltando.")
        return
    if columna_hora not in df_historico_referencia.columns:
        print(f" Error en 'corregir_dinamicos_con_estacional_simple': Columna '{columna_hora}' no encontrada en df_historico_referencia. Saltando.")
        return
    if columna_a_corregir not in df_historico_referencia.columns:
        print(f" Error en 'corregir_dinamicos_con_estacional_simple': Columna '{columna_a_corregir}' no en df_historico_referencia. Saltando.")
        return

    try:
        segmento_a_corregir_indices = df_a_corregir.loc[start_idx_anomalia:end_idx_anomalia].index
    except KeyError as e:
        print(f" Error de KeyError al intentar localizar segmento {start_idx_anomalia}-{end_idx_anomalia} en df_a_corregir (índice: {df_a_corregir.index.dtype}, tipo inicio: {type(start_idx_anomalia)}, tipo fin: {type(end_idx_anomalia)}). Error: {e}. Saltando.")
        return
    except Exception as e_segment:
        print(f" Error al obtener segmento {start_idx_anomalia}-{end_idx_anomalia} en df_a_corregir: {e_segment}. Saltando.")
        return

    for idx_anomalo in segmento_a_corregir_indices:
        if not isinstance(idx_anomalo, pd.Timestamp):
            continue

        # --- INICIO DE LA MODIFICACIÓN CLAVE PARA HORA_ANOMALA ---
        _hora_anomala_val_potencial_series = df_a_corregir.loc[idx_anomalo, columna_hora]
        if isinstance(_hora_anomala_val_potencial_series, pd.Series):
            if not _hora_anomala_val_potencial_series.empty:
                if _hora_anomala_val_potencial_series.nunique() > 1: # Chequeo opcional por consistencia
                    print(f" Advertencia: Múltiples horas diferentes ({_hora_anomala_val_potencial_series.unique()}) para el mismo timestamp {idx_anomalo} en df_a_corregir. Usando el primero: {_hora_anomala_val_potencial_series.iloc[0]}.")
                hora_anomala = _hora_anomala_val_potencial_series.iloc[0]
            else:
                # print(f"    Advertencia: Serie de hora anómala vacía para {idx_anomalo}. Saltando este punto.")
                continue
        else:
            hora_anomala = _hora_anomala_val_potencial_series

        fecha_anomala_dt_obj = idx_anomalo.date()

        target_past_dates = pd.date_range(end=fecha_anomala_dt_obj - pd.Timedelta(days=1), periods=ventana_dias_previos, freq='D').date

        min_lookup_date = fecha_anomala_dt_obj - pd.Timedelta(days=ventana_dias_previos + 5)
        max_lookup_date = fecha_anomala_dt_obj - pd.Timedelta(days=1)

        df_historico_relevante = df_historico_referencia[
            (df_historico_referencia.index.date >= min_lookup_date) &
            (df_historico_referencia.index.date <= max_lookup_date)
        ]

        if df_historico_relevante.empty:
            continue

        historico_index_dates = pd.Series(df_historico_relevante.index.date, index=df_historico_relevante.index)

        df_filtrado_temporal = df_historico_relevante[
            historico_index_dates.isin(target_past_dates) &
            (df_historico_relevante[columna_hora] == hora_anomala)
        ]

        valores_historicos_para_hora = df_filtrado_temporal[columna_a_corregir].dropna().tolist()

        if len(valores_historicos_para_hora) >= min_valores_historicos:
            if metodo_agregacion == 'mean':
                valor_corregido = np.mean(valores_historicos_para_hora)
            elif metodo_agregacion == 'median':
                valor_corregido = np.median(valores_historicos_para_hora)
            else:
                valor_corregido = np.nan

            if pd.notna(valor_corregido):
                df_a_corregir.loc[idx_anomalo, columna_a_corregir] = valor_corregido

# --- INICIO DEL PROCESO HÍBRIDO ---

df_entrada_para_atascos_nombre = 'df_corregido_faltantes'
var_X_model2_test_nombre = 'X_model2_test'
var_y_pred_model2_nombre = 'y_pred_model2'
var_label_encoder_model2_nombre = 'label_encoder_model2'
var_df_original_nombre = 'df_original'
var_columnas_objetivo_existentes_nombre = 'columnas_objetivo_existentes'
var_columnas_numericas_para_imputacion_nombre = 'columnas_numericas_para_imputacion'
nombre_clase_sensor_atascado_config = "Sensor Atascado"

if (df_entrada_para_atascos_nombre in locals() or df_entrada_para_atascos_nombre in globals()) and \
    (var_df_original_nombre in locals() or var_df_original_nombre in globals()) and \
    (var_columnas_objetivo_existentes_nombre in locals() or var_columnas_objetivo_existentes_nombre in globals()) and \
    (var_columnas_numericas_para_imputacion_nombre in locals() or var_columnas_numericas_para_imputacion_nombre in globals()):

    print("\n===============================================================================")
    print(f"INICIANDO FASE DE CORRECCIÓN HÍBRIDA PARA '{nombre_clase_sensor_atascado_config}'")
    print("===============================================================================")

    _tmp_entrada = locals().get(df_entrada_para_atascos_nombre, globals().get(df_entrada_para_atascos_nombre))
    df_entrada_correccion_actual_orig = _tmp_entrada.copy() if _tmp_entrada is not None else None
    _x_tmp = locals().get(var_X_model2_test_nombre, globals().get(var_X_model2_test_nombre))
    X_model2_test_actual_orig = _x_tmp.copy() if _x_tmp is not None else None
    y_pred_model2_actual = locals().get(var_y_pred_model2_nombre, globals().get(var_y_pred_model2_nombre))
    label_encoder_model2_actual = locals().get(var_label_encoder_model2_nombre, globals().get(var_label_encoder_model2_nombre))
    _tmp_original = locals().get(var_df_original_nombre, globals().get(var_df_original_nombre))
    df_original_eval_actual_orig = _tmp_original.copy() if _tmp_original is not None else None
    columnas_objetivo_actual = locals().get(var_columnas_objetivo_existentes_nombre, globals().get(var_columnas_objetivo_existentes_nombre))

    print("\n--- Fase de Preparación de Índices y Columna 'Hora' ---")

    df_original_eval_actual = None
    df_entrada_correccion_actual = None
    X_model2_test_actual = None

    # 1. df_original_eval_actual
    temp_df_original = df_original_eval_actual_orig.copy()
    if 'Fecha' in temp_df_original.columns:
        print(f"Procesando '{var_df_original_nombre}': Convirtiendo 'Fecha' a DatetimeIndex...")
        try:
            temp_df_original['Fecha'] = pd.to_datetime(temp_df_original['Fecha'])
            temp_df_original.set_index('Fecha', inplace=True)
            print(f"   Índice de '{var_df_original_nombre}' establecido a partir de 'Fecha'.")
        except Exception as e:
            print(f"   Error al convertir 'Fecha' para '{var_df_original_nombre}': {e}")
            
    if isinstance(temp_df_original.index, pd.DatetimeIndex):
        if 'Hora' not in temp_df_original.columns:
            temp_df_original['Hora'] = temp_df_original.index.hour
            print(f"   Columna 'Hora' creada en '{var_df_original_nombre}'.")
        df_original_eval_actual = temp_df_original
        
        if not df_original_eval_actual.index.is_unique:
            dupes_count = df_original_eval_actual.index.duplicated().sum()
            print(f"ADVERTENCIA: El índice de '{var_df_original_nombre}' (df_original_eval_actual) tiene {dupes_count} duplicados. Se eliminarán manteniendo la primera aparición.")
            df_original_eval_actual = df_original_eval_actual[~df_original_eval_actual.index.duplicated(keep='first')]
        if not df_original_eval_actual.index.is_monotonic_increasing: # implies sorted
            print(f"ADVERTENCIA: El índice de '{var_df_original_nombre}' (df_original_eval_actual) no es monotónicamente creciente. Se ordenará.")
            df_original_eval_actual = df_original_eval_actual.sort_index()
            
    else:
        print(f"ALERTA CRÍTICA: '{var_df_original_nombre}' NO tiene DatetimeIndex. La corrección estacional requiere esto.")
        df_original_eval_actual = temp_df_original

    # 2. df_entrada_correccion_actual
    temp_df_input = df_entrada_correccion_actual_orig.copy()
    if 'Fecha' in temp_df_input.columns:
        print(f"Procesando '{df_entrada_para_atascos_nombre}': Convirtiendo 'Fecha' a DatetimeIndex...")
        try:
            temp_df_input['Fecha'] = pd.to_datetime(temp_df_input['Fecha'])
            temp_df_input.set_index('Fecha', inplace=True)
            print(f"   Índice de '{df_entrada_para_atascos_nombre}' establecido a partir de 'Fecha'.")
        except Exception as e:
            print(f"   Error al convertir 'Fecha' para '{df_entrada_para_atascos_nombre}': {e}")

    if isinstance(temp_df_input.index, pd.DatetimeIndex):
        if 'Hora' not in temp_df_input.columns:
            temp_df_input['Hora'] = temp_df_input.index.hour
            print(f"   Columna 'Hora' creada en '{df_entrada_para_atascos_nombre}'.")
        df_entrada_correccion_actual = temp_df_input

        if not df_entrada_correccion_actual.index.is_unique:
            dupes_count_input = df_entrada_correccion_actual.index.duplicated().sum()
            print(f"ADVERTENCIA: El índice de '{df_entrada_para_atascos_nombre}' (df_entrada_correccion_actual) tiene {dupes_count_input} duplicados. Se eliminarán manteniendo la primera aparición.")
            df_entrada_correccion_actual = df_entrada_correccion_actual[~df_entrada_correccion_actual.index.duplicated(keep='first')]
        if not df_entrada_correccion_actual.index.is_monotonic_increasing:
            print(f"ADVERTENCIA: El índice de '{df_entrada_para_atascos_nombre}' (df_entrada_correccion_actual) no es monotónicamente creciente. Se ordenará.")
            df_entrada_correccion_actual = df_entrada_correccion_actual.sort_index()
            
    else:
        print(f"ALERTA CRÍTICA: '{df_entrada_para_atascos_nombre}' NO tiene DatetimeIndex. La corrección estacional requiere esto.")
        df_entrada_correccion_actual = temp_df_input

    # 3. X_model2_test_actual
    X_model2_test_actual = X_model2_test_actual_orig.copy() if X_model2_test_actual_orig is not None else None
    print(f"Procesando '{var_X_model2_test_nombre}':")
    if X_model2_test_actual is not None and not isinstance(X_model2_test_actual.index, pd.DatetimeIndex):
        print(f"   Índice de '{var_X_model2_test_nombre}' NO es DatetimeIndex. Intentando mapeo...")
        df_para_mapeo_fechas = df_original_eval_actual_orig.copy()

        if 'Fecha' in df_para_mapeo_fechas.columns:
            df_para_mapeo_fechas['Fecha_dt_temp'] = pd.to_datetime(df_para_mapeo_fechas['Fecha'])

            if X_model2_test_actual_orig.index.isin(df_para_mapeo_fechas.index).all():
                try:
                    fechas_mapeadas = df_para_mapeo_fechas.loc[X_model2_test_actual_orig.index, 'Fecha_dt_temp']
                    if not pd.Series(fechas_mapeadas).isnull().any():
                        X_model2_test_actual.index = pd.DatetimeIndex(fechas_mapeadas)
                        print(f"   Índice de '{var_X_model2_test_nombre}' establecido por mapeo a 'Fecha' de df_original.")
                        if 'Hora' not in X_model2_test_actual.columns and isinstance(X_model2_test_actual.index, pd.DatetimeIndex):
                            X_model2_test_actual['Hora'] = X_model2_test_actual.index.hour
                            print(f"   Columna 'Hora' (re)creada en '{var_X_model2_test_nombre}'.")
                    else:
                        print(f"   FALLO el mapeo para '{var_X_model2_test_nombre}': algunas fechas mapeadas resultaron NaT.")
                except KeyError:
                    print(f"   FALLO el mapeo para '{var_X_model2_test_nombre}': los valores del índice de X_test ({X_model2_test_actual_orig.index.dtype}) no se encontraron como índice en df_original ({df_para_mapeo_fechas.index.dtype}).")
                except Exception as e:
                    print(f"   Error complejo durante el mapeo de índice para '{var_X_model2_test_nombre}': {e}")
            else:
                print(f"   ALERTA: No todos los índices de '{var_X_model2_test_nombre}' se encuentran en el índice de 'df_original_eval_actual_orig'. Mapeo no posible de esta forma.")
                # faltantes_idx = X_model2_test_actual_orig.index[~X_model2_test_actual_orig.index.isin(df_para_mapeo_fechas.index)]
                # print(f"     Ejemplo de índices de X_test no encontrados en df_original_eval_actual_orig: {faltantes_idx[:5].tolist()}...")
        else:
            print(f"   ALERTA: No se puede realizar el mapeo para '{var_X_model2_test_nombre}'. 'df_original_eval_actual_orig' no tiene columna 'Fecha'.")

    if X_model2_test_actual is not None and not isinstance(X_model2_test_actual.index, pd.DatetimeIndex):
        print(f"ALERTA FINAL: '{var_X_model2_test_nombre}' AÚN NO tiene DatetimeIndex. La alineación de predicciones y la corrección estacional probablemente fallarán.")

    print("--- Fin de Fase de Preparación de Índices ---")

    if df_entrada_correccion_actual is None:
        print(f"ALERTA CRÍTICA: df_entrada_correccion_actual es None. Usando la copia original df_entrada_correccion_actual_orig.")
        df_entrada_correccion_actual = df_entrada_correccion_actual_orig.copy()

    df_stuck_hybrid_corrected = df_entrada_correccion_actual.copy()

    if 'pred_tipo_anomalia' in df_stuck_hybrid_corrected.columns:
        serie_predicciones_hybrid = df_stuck_hybrid_corrected['pred_tipo_anomalia']
        print("\n✅ Columna 'pred_tipo_anomalia' encontrada. Se filtrarán solo los predecidos como Atascos.")
    else:
        # Intentamos usar las variables de predicción guardadas si existen
        if (y_pred_model2_actual is not None) and (label_encoder_model2_actual is not None) and (X_model2_test_actual is not None) and (X_model2_test_actual.index is not None):
            try:
                serie_pred_temp = pd.Series(
                    label_encoder_model2_actual.inverse_transform(y_pred_model2_actual),
                    index=X_model2_test_actual.index
                )
                serie_predicciones_hybrid = pd.Series(dtype='object', index=df_stuck_hybrid_corrected.index)
                common_idx = df_stuck_hybrid_corrected.index.intersection(serie_pred_temp.index)
                serie_predicciones_hybrid.loc[common_idx] = serie_pred_temp.loc[common_idx]
                print("\n✅ Se usaron las variables 'y_pred_model2_actual' para filtrar Atascos.")
            except Exception as e:
                # Fallback en caso de error
                serie_predicciones_hybrid = pd.Series(nombre_clase_sensor_atascado_config, index=df_stuck_hybrid_corrected.index)
                print(f"\n⚠️ Falló la carga de variables de predicción ({e}). Se usará pura heurística.")
        else:
            # Fallback final: Si no hay forma de saber las predicciones, asumimos todo como Sensor Atascado 
            # (así la función marcar_segmentos_stuck... funcionará usando solo la heurística de repetición)
            serie_predicciones_hybrid = pd.Series(nombre_clase_sensor_atascado_config, index=df_stuck_hybrid_corrected.index)
            print("\n⚠️ No se encontraron predicciones. Se marcarán Atascos usando puramente la heurística de repetición matemática.")

    min_duracion_stuck_hybrid = 5

    columnas_dinamicas_stuck = ['PRAD', 'PRGINT']  # v3: UVENT_cen y UVENT_lN excluidas
    columnas_dinamicas_stuck = [
        col for col in columnas_dinamicas_stuck
        if col in df_stuck_hybrid_corrected.columns and col in columnas_objetivo_actual
    ]
    print(f"Columnas dinámicas para corrección de atascos: {columnas_dinamicas_stuck}")

    segmentos_nanificados_dinamicos_eval = []

    print(f"\n--- Marcando como NaN Segmentos Atascados en Columnas Dinámicas (min_duracion={min_duracion_stuck_hybrid}) ---")
    for col_dyn in columnas_dinamicas_stuck:
        if col_dyn not in df_stuck_hybrid_corrected.columns:
            print(f"Advertencia: Columna dinámica '{col_dyn}' no encontrada. Saltando marcado.")
            continue
        if serie_predicciones_hybrid.empty or serie_predicciones_hybrid.isnull().all():
            print(f"Advertencia: 'serie_predicciones_hybrid' está vacía o toda NaN para la columna '{col_dyn}'. Saltando marcado.")
            continue

        num_marcados_col, lista_segmentos_col = marcar_segmentos_stuck_como_nan_condicional(
            df_stuck_hybrid_corrected,
            col_dyn,
            serie_predicciones_hybrid,
            nombre_clase_sensor_atascado=nombre_clase_sensor_atascado_config,
            min_duracion_atasco=min_duracion_stuck_hybrid
        )
        if num_marcados_col > 0:
            print(f"   En columna DINÁMICA '{col_dyn}': {num_marcados_col} segmentos marcados como NaN.")
            for seg_start, seg_end in lista_segmentos_col:
                segmentos_nanificados_dinamicos_eval.append((col_dyn, seg_start, seg_end))

    if segmentos_nanificados_dinamicos_eval:
        print(f"\nValores NaN ANTES de Corrección Estacional para atascos dinámicos (columnas dinámicas objetivo):")
        nan_sum_before_seasonal = df_stuck_hybrid_corrected[columnas_dinamicas_stuck].isnull().sum()
        print(nan_sum_before_seasonal[nan_sum_before_seasonal > 0])

        print("\n--- Corrigiendo Atascos en Columnas Dinámicas con Método Estacional Simple ---")
        ventana_dias_previos_seasonal = 7
        metodo_agregacion_seasonal = 'median'
        min_valores_historicos_seasonal = 1
        columna_hora_nombre = 'Hora'
        
        if df_original_eval_actual is None or not isinstance(df_original_eval_actual.index, pd.DatetimeIndex):
            print("ALERTA CRÍTICA: df_original_eval_actual no es válido para la corrección estacional. Saltando la corrección de dinámicos.")
        else:
            for col_dyn_corregir, seg_start_idx, seg_end_idx in segmentos_nanificados_dinamicos_eval:
                print(f"   Corrigiendo columna dinámica '{col_dyn_corregir}' para el segmento de {seg_start_idx} a {seg_end_idx}...")
                corregir_dinamicos_con_estacional_simple(
                    df_a_corregir=df_stuck_hybrid_corrected,
                    df_historico_referencia=df_original_eval_actual,
                    columna_a_corregir=col_dyn_corregir,
                    start_idx_anomalia=seg_start_idx,
                    end_idx_anomalia=seg_end_idx,
                    columna_hora=columna_hora_nombre,
                    ventana_dias_previos=ventana_dias_previos_seasonal,
                    metodo_agregacion=metodo_agregacion_seasonal,
                    min_valores_historicos=min_valores_historicos_seasonal
                )
        
            print("Corrección estacional simple para columnas dinámicas completada.")
            nan_sum_after_seasonal = df_stuck_hybrid_corrected[columnas_dinamicas_stuck].isnull().sum()
            print(f"NaNs DESPUÉS de corrección estacional (en columnas dinámicas objetivo):")
            print(nan_sum_after_seasonal[nan_sum_after_seasonal > 0])

        # --- Bloque de Evaluación Estacional ---
        all_true_stuck_seasonal_dinamicos_eval = []
        all_imputed_stuck_seasonal_dinamicos_eval = []
        column_stuck_correction_metrics_seasonal_eval = []

        if df_original_eval_actual is None or not isinstance(df_original_eval_actual.index, pd.DatetimeIndex):
            print("ALERTA CRÍTICA: df_original_eval_actual no es válido para la evaluación estacional. Saltando la evaluación.")
        elif segmentos_nanificados_dinamicos_eval:
            print("\n--- Evaluación Detallada de Corrección (Estacional Simple) para Sensores Dinámicos Atascados ---")
            for col_eval, start_idx_eval, end_idx_eval in segmentos_nanificados_dinamicos_eval:
                if not (start_idx_eval in df_original_eval_actual.index and \
                        end_idx_eval in df_original_eval_actual.index and \
                        start_idx_eval in df_stuck_hybrid_corrected.index and \
                        end_idx_eval in df_stuck_hybrid_corrected.index and \
                        col_eval in df_original_eval_actual.columns and \
                        col_eval in df_stuck_hybrid_corrected.columns):
                    # print(f"Advertencia (Evaluación Estacional): Segmento ({col_eval}, {start_idx_eval}-{end_idx_eval}) tiene índices o columna no válidos. Saltando.")
                    continue
                
                # The problematic line:
                true_vals_seg_eval = df_original_eval_actual.loc[start_idx_eval:end_idx_eval, col_eval]
                imputed_vals_seg_eval = df_stuck_hybrid_corrected.loc[start_idx_eval:end_idx_eval, col_eval]
                
                if len(true_vals_seg_eval) == len(imputed_vals_seg_eval) and true_vals_seg_eval.notna().all():
                    valid_imputed_mask = imputed_vals_seg_eval.notna()
                    # if not valid_imputed_mask.all():
                        # print(f"Info (Evaluación Estacional): Valores imputados para {col_eval} en segmento {start_idx_eval}-{end_idx_eval} contienen NaNs DESPUÉS de imputación estacional. Excluyendo NaNs de métricas para este segmento.")
                        # pass
                    true_vals_seg_for_metric = true_vals_seg_eval[valid_imputed_mask]
                    imputed_vals_seg_for_metric = imputed_vals_seg_eval[valid_imputed_mask]
                    if imputed_vals_seg_for_metric.empty:
                        # print(f"  Segmento {col_eval} ({start_idx_eval}-{end_idx_eval}) no tiene valores válidos post-imputación para métricas.")
                        continue
                    all_true_stuck_seasonal_dinamicos_eval.extend(true_vals_seg_for_metric.tolist())
                    all_imputed_stuck_seasonal_dinamicos_eval.extend(imputed_vals_seg_for_metric.tolist())
                    rmse_seg_seasonal = np.sqrt(mean_squared_error(true_vals_seg_for_metric, imputed_vals_seg_for_metric))
                    mae_seg_seasonal = mean_absolute_error(true_vals_seg_for_metric, imputed_vals_seg_for_metric)
                    try:
                        start_loc = df_original_eval_actual.index.get_loc(start_idx_eval)
                        end_loc = df_original_eval_actual.index.get_loc(end_idx_eval)
                        if isinstance(start_loc, slice): start_loc = start_loc.start
                        if isinstance(end_loc, slice): end_loc = end_loc.stop -1
                        segmento_filas_str = f"{start_loc}-{end_loc}"
                    except (KeyError, TypeError):
                        segmento_filas_str = f"Idx:{start_idx_eval}-Idx:{end_idx_eval}"
                    column_stuck_correction_metrics_seasonal_eval.append({
                        'Columna': col_eval, 'Segmento_Filas': segmento_filas_str,
                        'Num_Valores_Corregidos': len(true_vals_seg_for_metric),
                        'RMSE_Correccion_Seasonal': rmse_seg_seasonal, 'MAE_Correccion_Seasonal': mae_seg_seasonal,
                        'Media_Real_Segmento': true_vals_seg_for_metric.mean(), 'Media_Imputada_Segmento': imputed_vals_seg_for_metric.mean()
                    })
                elif not true_vals_seg_eval.notna().all():
                    # print(f"Advertencia (Evaluación Estacional): Valores verdaderos para {col_eval} en segmento {start_idx_eval}-{end_idx_eval} contienen NaNs en df_original_eval_actual. Saltando este segmento.")
                    pass
            if column_stuck_correction_metrics_seasonal_eval:
                df_stuck_eval_metrics_seasonal = pd.DataFrame(column_stuck_correction_metrics_seasonal_eval)
                resumen_por_columna_seasonal_stuck = df_stuck_eval_metrics_seasonal.groupby('Columna').agg(
                    Total_Segmentos_Corregidos = ('Segmento_Filas', 'count'),
                    Promedio_RMSE_Seasonal = ('RMSE_Correccion_Seasonal', 'mean'),
                    Promedio_MAE_Seasonal = ('MAE_Correccion_Seasonal', 'mean'),
                    Suma_Valores_Corregidos = ('Num_Valores_Corregidos', 'sum')
                ).reset_index()
                print("\nMétricas de Calidad de Corrección (Estacional Simple en Dinámicos) - Resumen por Columna:")
                print(resumen_por_columna_seasonal_stuck.to_string())

                if all_true_stuck_seasonal_dinamicos_eval and all_imputed_stuck_seasonal_dinamicos_eval:
                    valid_true_global = pd.Series(all_true_stuck_seasonal_dinamicos_eval)
                    valid_imputed_global = pd.Series(all_imputed_stuck_seasonal_dinamicos_eval)
                    nan_mask_global = valid_true_global.notna() & valid_imputed_global.notna()
                    valid_true_global_clean = valid_true_global[nan_mask_global]
                    valid_imputed_global_clean = valid_imputed_global[nan_mask_global]
                    if not valid_true_global_clean.empty:
                        global_rmse_seasonal = np.sqrt(mean_squared_error(valid_true_global_clean, valid_imputed_global_clean))
                        global_mae_seasonal = mean_absolute_error(valid_true_global_clean, valid_imputed_global_clean)
                        print(f"\n  RMSE Global para Dinámicos (Corrección Estacional): {global_rmse_seasonal:.3f} (sobre {len(valid_true_global_clean)} puntos)")
                        print(f"  MAE Global para Dinámicos (Corrección Estacional): {global_mae_seasonal:.3f} (sobre {len(valid_true_global_clean)} puntos)")
                        plt.figure(figsize=(8, 8))
                        sample_size_stuck_seasonal = min(len(valid_true_global_clean), 2000)
                        if sample_size_stuck_seasonal > 0:
                            indices_muestreados = np.random.choice(valid_true_global_clean.index, size=sample_size_stuck_seasonal, replace=False)
                            true_sampled_values = valid_true_global_clean.loc[indices_muestreados].values
                            imputed_sampled_values = valid_imputed_global_clean.loc[indices_muestreados].values
                            plt.scatter(
                                true_sampled_values,
                                imputed_sampled_values,
                                alpha=0.5, s=10, label='Corrected Points'
                            )
                        min_val_plot = valid_true_global_clean.min()
                        max_val_plot = valid_true_global_clean.max()
                        if not valid_imputed_global_clean.empty:
                            min_val_plot = min(min_val_plot, valid_imputed_global_clean.min())
                            max_val_plot = max(max_val_plot, valid_imputed_global_clean.max())
                        if pd.notna(min_val_plot) and pd.notna(max_val_plot) and min_val_plot != max_val_plot:
                            plt.plot([min_val_plot, max_val_plot], [min_val_plot, max_val_plot], 'r--', lw=2, label='Ideal')
                        plt.xlabel("True Values")
                        plt.ylabel("Corrected Values")
                        plt.legend()
                        plt.grid(True)
                        plt.tight_layout()
                        plt.show()
                    else:
                        print("\nNo hay suficientes datos válidos globales (después de filtrar NaNs) para calcular RMSE/MAE o graficar para la corrección estacional.")
            else:
                print("\nNo se evaluaron segmentos de 'Sensor Atascado' en dinámicos con Corrección Estacional (lista de métricas vacía).")
        else:
            print("\nNo se encontraron segmentos marcados como NaN para corrección de atasco en dinámicos (lista 'segmentos_nanificados_dinamicos_eval' vacía después del marcado inicial).")
    else:
        print("\nNo se marcaron segmentos en columnas dinámicas para corrección estacional (lista 'segmentos_nanificados_dinamicos_eval' vacía al inicio).")

    # --- Parte 2: Corrección con Interpolación Lineal para Sensores Estables Atascados ---
    columnas_estables_stuck = [
        col for col in columnas_objetivo_actual
        if col not in columnas_dinamicas_stuck and col in df_stuck_hybrid_corrected.columns
    ]
    print(f"\nColumnas estables para corrección de atascos por interpolación: {columnas_estables_stuck}")
    segmentos_interpolados_estables_eval = []

    print(f"\n--- Iniciando Corrección por Interpolación para '{nombre_clase_sensor_atascado_config}' en Sensores Estables (min_duracion={min_duracion_stuck_hybrid}) ---")
    if columnas_estables_stuck:
        for col_stab in columnas_estables_stuck:
            if col_stab not in df_stuck_hybrid_corrected.columns:
                print(f"Advertencia: Columna estable '{col_stab}' no encontrada en df_stuck_hybrid_corrected. Saltando interpolación.")
                continue
            if serie_predicciones_hybrid.empty or serie_predicciones_hybrid.isnull().all():
                print(f"Advertencia: 'serie_predicciones_hybrid' está vacía o toda NaN para columna estable '{col_stab}'. Saltando interpolación.")
                continue

            temp_serie_predicciones_alineada_stable = pd.Series(dtype=object)
            if isinstance(df_stuck_hybrid_corrected.index, pd.DatetimeIndex) and \
                isinstance(serie_predicciones_hybrid.index, pd.DatetimeIndex):
                if df_stuck_hybrid_corrected.index.equals(serie_predicciones_hybrid.index):
                    temp_serie_predicciones_alineada_stable = serie_predicciones_hybrid
                else:
                    common_idx_pred_stable = df_stuck_hybrid_corrected.index.intersection(serie_predicciones_hybrid.index)
                    if not common_idx_pred_stable.empty:
                        temp_serie_predicciones_alineada_stable = serie_predicciones_hybrid.loc[common_idx_pred_stable].reindex(df_stuck_hybrid_corrected.index)
                    else:
                        print(f"Advertencia: No hay índices Datetime comunes para alinear 'serie_predicciones_hybrid' con 'df_stuck_hybrid_corrected' para la columna estable '{col_stab}'. Saltando interpolación.")
                        continue
            else:
                print(f"Advertencia: Índices no son ambos DatetimeIndex para alinear 'serie_predicciones_hybrid' con 'df_stuck_hybrid_corrected' para la columna estable '{col_stab}'. Saltando interpolación.")
                continue

            if temp_serie_predicciones_alineada_stable.isnull().all():
                print(f"Advertencia: 'serie_predicciones_hybrid' (después de intentar alinear) está toda NaN para la columna estable '{col_stab}'. Saltando interpolación.")
                continue

            num_corregidos_col, lista_segmentos_col = corregir_sensor_atascado_condicional_eval(
                df_stuck_hybrid_corrected,
                col_stab,
                temp_serie_predicciones_alineada_stable,
                nombre_clase_sensor_atascado=nombre_clase_sensor_atascado_config,
                min_duracion_atasco=min_duracion_stuck_hybrid
            )
            if num_corregidos_col > 0:
                print(f"   En columna ESTABLE '{col_stab}': {num_corregidos_col} segmentos marcados como NaN e interpolados.")
                for seg_start, seg_end in lista_segmentos_col:
                    segmentos_interpolados_estables_eval.append((col_stab, seg_start, seg_end))

        all_true_stuck_stable_eval = []
        all_interpolated_stuck_stable_eval = []
        column_stuck_correction_metrics_stable_eval = []
        
        if df_original_eval_actual is None or not isinstance(df_original_eval_actual.index, pd.DatetimeIndex):
            print("ALERTA CRÍTICA: df_original_eval_actual no es válido para la evaluación de interpolación. Saltando la evaluación.")
        elif segmentos_interpolados_estables_eval:
            print("\n--- Evaluación Detallada de Corrección (Interpolación Lineal) para Sensores Estables Atascados ---")
            for col_eval, start_idx_eval, end_idx_eval in segmentos_interpolados_estables_eval:
                if not (start_idx_eval in df_original_eval_actual.index and \
                        end_idx_eval in df_original_eval_actual.index and \
                        start_idx_eval in df_stuck_hybrid_corrected.index and \
                        end_idx_eval in df_stuck_hybrid_corrected.index and \
                        col_eval in df_original_eval_actual.columns and \
                        col_eval in df_stuck_hybrid_corrected.columns):
                    print(f"Advertencia (Evaluación Interpolación): Segmento ({col_eval}, {start_idx_eval}-{end_idx_eval}) tiene índices o columna no válidos. Saltando.")
                    continue
                true_vals_seg_eval = df_original_eval_actual.loc[start_idx_eval:end_idx_eval, col_eval]
                interpolated_vals_seg_eval = df_stuck_hybrid_corrected.loc[start_idx_eval:end_idx_eval, col_eval]
                if len(true_vals_seg_eval) == len(interpolated_vals_seg_eval) and true_vals_seg_eval.notna().all():
                    valid_interp_mask = interpolated_vals_seg_eval.notna()
                    # if not valid_interp_mask.all():
                        # print(f"Info (Evaluación Interpolación): Valores para {col_eval} en segmento {start_idx_eval}-{end_idx_eval} contienen NaNs DESPUÉS de interpolación. Excluyendo NaNs de métricas.")
                    true_vals_seg_for_metric_stable = true_vals_seg_eval[valid_interp_mask]
                    interpolated_vals_seg_for_metric_stable = interpolated_vals_seg_eval[valid_interp_mask]
                    if interpolated_vals_seg_for_metric_stable.empty:
                        print(f"  Segmento {col_eval} ({start_idx_eval}-{end_idx_eval}) no tiene valores válidos post-interpolación para métricas.")
                        continue
                    all_true_stuck_stable_eval.extend(true_vals_seg_for_metric_stable.tolist())
                    all_interpolated_stuck_stable_eval.extend(interpolated_vals_seg_for_metric_stable.tolist())
                    rmse_seg_stable = np.sqrt(mean_squared_error(true_vals_seg_for_metric_stable, interpolated_vals_seg_for_metric_stable))
                    mae_seg_stable = mean_absolute_error(true_vals_seg_for_metric_stable, interpolated_vals_seg_for_metric_stable)
                    try:
                        start_loc = df_original_eval_actual.index.get_loc(start_idx_eval)
                        end_loc = df_original_eval_actual.index.get_loc(end_idx_eval)
                        if isinstance(start_loc, slice): start_loc = start_loc.start
                        if isinstance(end_loc, slice): end_loc = end_loc.stop -1
                        segmento_filas_str = f"{start_loc}-{end_loc}"
                    except (KeyError, TypeError):
                        segmento_filas_str = f"Idx:{start_idx_eval}-Idx:{end_idx_eval}"
                    column_stuck_correction_metrics_stable_eval.append({
                        'Columna': col_eval,
                        'Segmento_Filas': segmento_filas_str,
                        'Num_Valores_Corregidos': len(true_vals_seg_for_metric_stable),
                        'RMSE_Correccion_Interp': rmse_seg_stable,
                        'MAE_Correccion_Interp': mae_seg_stable,
                        'Media_Real_Segmento': true_vals_seg_for_metric_stable.mean(),
                        'Media_Interpolada_Segmento': interpolated_vals_seg_for_metric_stable.mean()
                    })
                elif not true_vals_seg_eval.notna().all():
                    print(f"Advertencia (Evaluación Interpolación): Valores verdaderos para {col_eval} en segmento {start_idx_eval}-{end_idx_eval} contienen NaNs en df_original_eval_actual. Saltando.")
            if column_stuck_correction_metrics_stable_eval:
                df_stuck_eval_metrics_stable = pd.DataFrame(column_stuck_correction_metrics_stable_eval)
                resumen_por_columna_stable_stuck = df_stuck_eval_metrics_stable.groupby('Columna').agg(
                    Total_Segmentos_Corregidos = ('Segmento_Filas', 'count'),
                    Promedio_RMSE_Interp = ('RMSE_Correccion_Interp', 'mean'),
                    Promedio_MAE_Interp = ('MAE_Correccion_Interp', 'mean'),
                    Suma_Valores_Corregidos = ('Num_Valores_Corregidos', 'sum')
                ).reset_index()
                print("\nMétricas de Calidad de Corrección (Interpolación en Estables) - Resumen por Columna:")
                print(resumen_por_columna_stable_stuck.to_string())
                if all_true_stuck_stable_eval and all_interpolated_stuck_stable_eval:
                    valid_true_stable_global = pd.Series(all_true_stuck_stable_eval)
                    valid_interpolated_stable_global = pd.Series(all_interpolated_stuck_stable_eval)
                    nan_mask_stable_global = valid_true_stable_global.notna() & valid_interpolated_stable_global.notna()
                    valid_true_stable_global_clean = valid_true_stable_global[nan_mask_stable_global]
                    valid_interpolated_stable_global_clean = valid_interpolated_stable_global[nan_mask_stable_global]
                    if not valid_true_stable_global_clean.empty:
                        global_rmse_stable = np.sqrt(mean_squared_error(valid_true_stable_global_clean, valid_interpolated_stable_global_clean))
                        global_mae_stable = mean_absolute_error(valid_true_stable_global_clean, valid_interpolated_stable_global_clean)
                        print(f"\n  RMSE Global para Estables (Interpolación): {global_rmse_stable:.3f} (sobre {len(valid_true_stable_global_clean)} puntos)")
                        print(f"  MAE Global para Estables (Interpolación): {global_mae_stable:.3f} (sobre {len(valid_true_stable_global_clean)} puntos)")
                        plt.figure(figsize=(8, 8))
                        sample_size_stuck_stable = min(len(valid_true_stable_global_clean), 2000)
                        if sample_size_stuck_stable > 0:
                            indices_muestreados_stable = np.random.choice(valid_true_stable_global_clean.index, size=sample_size_stuck_stable, replace=False)
                            true_sampled_stable_values = valid_true_stable_global_clean.loc[indices_muestreados_stable].values
                            interpolated_sampled_stable_values = valid_interpolated_stable_global_clean.loc[indices_muestreados_stable].values
                            plt.scatter(
                                true_sampled_stable_values,
                                interpolated_sampled_stable_values,
                                alpha=0.5, s=10, label='Corrected Points', c='green'
                            )
                        min_val_plot_stable = valid_true_stable_global_clean.min()
                        max_val_plot_stable = valid_true_stable_global_clean.max()
                        if not valid_interpolated_stable_global_clean.empty:
                            min_val_plot_stable = min(min_val_plot_stable, valid_interpolated_stable_global_clean.min())
                            max_val_plot_stable = max(max_val_plot_stable, valid_interpolated_stable_global_clean.max())
                        if pd.notna(min_val_plot_stable) and pd.notna(max_val_plot_stable) and min_val_plot_stable != max_val_plot_stable:
                            plt.plot([min_val_plot_stable, max_val_plot_stable], [min_val_plot_stable, max_val_plot_stable], 'r--', lw=2, label='Ideal')
                        plt.xlabel("True Values")
                        plt.ylabel("Corrected Values")
                        plt.legend()
                        plt.grid(True)
                        plt.tight_layout()
                        plt.show()
                    else:
                        print("\nNo hay suficientes datos válidos globales (después de filtrar NaNs) para calcular RMSE/MAE o graficar para la corrección por interpolación.")
            else:
                print("\nNo se evaluaron segmentos de 'Sensor Atascado' en estables con Interpolación (lista de métricas vacía).")
        else:
            print("\nNo se encontraron segmentos marcados e interpolados para evaluación en sensores estables.")
    else:
        print("\nNo hay columnas de sensores estables para corregir atascos con interpolación o no se encontraron atascos que cumplan criterios.")

    if 'temp_M2_pred_type_hybrid' in df_stuck_hybrid_corrected.columns:
        df_stuck_hybrid_corrected.drop(columns=['temp_M2_pred_type_hybrid'], inplace=True)

    print("\n===============================================================================")
    print(f"FIN DE LA FASE DE CORRECCIÓN HÍBRIDA PARA '{nombre_clase_sensor_atascado_config}'")
    print("===============================================================================")

    print("\nEl DataFrame con las correcciones de 'Datos Faltantes' y 'Sensor Atascado' (usando método estacional para dinámicos) se encuentra en: df_stuck_hybrid_corrected")

else:
    print("Error: DataFrames o variables necesarios para la corrección HÍBRIDA de 'Sensor Atascado' no están disponibles en el entorno global o local.")
    print("Por favor, verifica los siguientes NOMBRES DE VARIABLES en tu notebook y asegúrate de que estén definidos ANTES de ejecutar este bloque:")
    print(f"  - DataFrame de entrada (post-imputación faltantes): '{df_entrada_para_atascos_nombre}'")
    print(f"  - X_test de tu modelo de anomalías: '{var_X_model2_test_nombre}'")
    print(f"  - Predicciones de tu modelo de anomalías: '{var_y_pred_model2_nombre}'")
    print(f"  - LabelEncoder de tu modelo de anomalías: '{var_label_encoder_model2_nombre}'")
    print(f"  - DataFrame original para evaluación: '{var_df_original_nombre}'")
    print(f"  - Lista de columnas objetivo: '{var_columnas_objetivo_existentes_nombre}'")
    print(f"  - Lista de columnas para imputación general (referencia): '{var_columnas_numericas_para_imputacion_nombre}'")

# This part might error if df_stuck_hybrid_corrected was not successfully defined due to issues above
if 'df_stuck_hybrid_corrected' in locals() or 'df_stuck_hybrid_corrected' in globals():
    if columnas_objetivo_actual and all(col in df_stuck_hybrid_corrected.columns for col in columnas_objetivo_actual if isinstance(col, str)): # check if list is not empty and cols exist
        print("\nValores NaN DESPUÉS de la corrección híbrida completa:")
        nans_despues_hibrido = df_stuck_hybrid_corrected[columnas_objetivo_actual].isnull().sum()
        print(nans_despues_hibrido[nans_despues_hibrido > 0])
        # columnas_con_nans_restantes = nans_despues_hibrido[nans_despues_hibrido > 0].index.tolist() # Not used later, commented
    else:
        print("\nNo se pueden mostrar los NaNs después de la corrección: 'columnas_objetivo_actual' no definidas o no válidas para 'df_stuck_hybrid_corrected'.")
else:
    print("\n'df_stuck_hybrid_corrected' no fue definido, no se pueden mostrar los NaNs finales.")

In [ ]:
if 'df_stuck_hybrid_corrected' in locals() or 'df_stuck_hybrid_corrected' in globals():
    print("\nValores NaN DESPUÉS de la corrección híbrida completa:")
    nans_despues_hibrido = df_stuck_hybrid_corrected[columnas_objetivo_existentes].isnull().sum()
    print(nans_despues_hibrido[nans_despues_hibrido > 0])
    columnas_con_nans_restantes = nans_despues_hibrido[nans_despues_hibrido > 0].index.tolist()
else:
    print("'df_stuck_hybrid_corrected' no disponible. Saltando resumen de NaNs.")
    columnas_con_nans_restantes = []

In [ ]:
current_df_to_clean = locals().get('df_stuck_hybrid_corrected', globals().get('df_stuck_hybrid_corrected'))  # None si no está definido

if 'iterative_imputer_faltantes' in locals() and iterative_imputer_faltantes is not None and current_df_to_clean is not None:
    # --- PASO FINAL DE IMPUTACIÓN Y EVALUACIÓN ---
    print(f"\n===============================================================================")
    print(f"INICIANDO FASE DE IMPUTACIÓN FINAL Y EVALUACIÓN")
    print(f"===============================================================================")

    # Identificar las columnas que realmente tienen NaNs y que el imputador maneja.
    # 'columnas_numericas_para_imputacion' debe ser la lista de columnas con la que 'iterative_imputer_faltantes' fue originalmente AJUSTADO (fit).
    if not ('columnas_numericas_para_imputacion' in locals() and columnas_numericas_para_imputacion):
        print("Error: 'columnas_numericas_para_imputacion' no está definida o está vacía. No se puede continuar.")
    else:
        # Columnas que el imputador espera
        cols_for_imputer_transform = [col for col in columnas_numericas_para_imputacion if col in current_df_to_clean.columns]
        
        if not cols_for_imputer_transform:
            print(f"Advertencia: Ninguna de las 'columnas_numericas_para_imputacion' se encontró en el DataFrame actual.")
        else:
            nan_check_df = current_df_to_clean[cols_for_imputer_transform]
            nans_before_final_imputation_series = nan_check_df.isnull().sum()
            columnas_con_nans_para_este_paso = nans_before_final_imputation_series[nans_before_final_imputation_series > 0].index.tolist()

            if columnas_con_nans_para_este_paso:
                print(f"\nColumnas con NaNs restantes que serán procesadas por IterativeImputer: {columnas_con_nans_para_este_paso}")

                # --- Preparación para la evaluación ---
                nan_indices_por_columna_final_eval = {}
                for col_eval_final in columnas_con_nans_para_este_paso:
                    # Guardar índices donde la columna es NaN ANTES de la imputación de este paso
                    nan_indices_por_columna_final_eval[col_eval_final] = current_df_to_clean[current_df_to_clean[col_eval_final].isnull()].index
                
                # --- Aplicar la imputación ---
                # .transform() espera TODAS las columnas con las que el imputador fue ajustado (fit), en el mismo orden.
                print(f"\nAplicando IterativeImputer.transform() a {len(cols_for_imputer_transform)} columnas.")
                
                # Prevenir el SettingWithCopyWarning obteniendo los valores y reasignando
                imputed_values_array = iterative_imputer_faltantes.transform(current_df_to_clean[cols_for_imputer_transform])
                imputed_subset_df = pd.DataFrame(imputed_values_array, index=current_df_to_clean.index, columns=cols_for_imputer_transform)
                
                for col_update in cols_for_imputer_transform:
                    current_df_to_clean[col_update] = imputed_subset_df[col_update]

                print("IterativeImputer para imputación final completado.")
                nans_final_count = current_df_to_clean[columnas_con_nans_para_este_paso].isnull().sum().sum()
                print(f"Total NaNs en estas columnas después del paso final: {nans_final_count}")

                # --- Evaluación de esta imputación final ---
                df_original_ref_final = locals().get('df_original_eval_actual', globals().get('df_original_eval_actual'))
                if df_original_ref_final is None:
                    # Fallback al df_original global si no se encuentra la versión evaluable
                    df_original_ref_final = locals().get('df_original', globals().get('df_original'))

                if df_original_ref_final is not None and nan_indices_por_columna_final_eval:
                    print("\n--- Evaluación Detallada de la Imputación Final ---")
                    all_true_final_eval = []
                    all_imputed_final_eval = []
                    final_imputation_metrics_eval = []
                    
                    for col_eval_final, nan_indices_final in nan_indices_por_columna_final_eval.items():
                        if nan_indices_final.empty:
                            continue
                        
                        # Intersección para asegurar que los índices existen en ambos DataFrames
                        valid_indices_for_eval = df_original_ref_final.index.intersection(nan_indices_final)
                        
                        if not (col_eval_final in df_original_ref_final.columns and col_eval_final in current_df_to_clean.columns):
                            print(f"Advertencia (Eval Final): Columna '{col_eval_final}' no consistente entre DFs. Saltando.")
                            continue
                        if valid_indices_for_eval.empty:
                            # print(f"Info (Eval Final): No hay índices comunes para '{col_eval_final}' para evaluación. Saltando.")
                            continue

                        true_values_final = df_original_ref_final.loc[valid_indices_for_eval, col_eval_final]
                        imputed_values_final = current_df_to_clean.loc[valid_indices_for_eval, col_eval_final] # Valores DESPUÉS de la imputación

                        valid_comparison_mask_final = true_values_final.notna() & imputed_values_final.notna()
                        true_values_valid_final = true_values_final[valid_comparison_mask_final]
                        imputed_values_valid_final = imputed_values_final[valid_comparison_mask_final]

                        if not true_values_valid_final.empty:
                            all_true_final_eval.extend(true_values_valid_final.tolist())
                            all_imputed_final_eval.extend(imputed_values_valid_final.tolist())

                            rmse_final_col = np.sqrt(mean_squared_error(true_values_valid_final, imputed_values_valid_final))
                            mae_final_col = mean_absolute_error(true_values_valid_final, imputed_values_valid_final)

                            final_imputation_metrics_eval.append({
                                'Columna': col_eval_final,
                                'Num_Valores_Imputados_Eval': len(true_values_valid_final),
                                'RMSE': rmse_final_col,
                                'MAE': mae_final_col,
                                'Media_Real_Original': true_values_valid_final.mean(),
                                'Media_Imputada_Final': imputed_values_valid_final.mean()
                            })
                    
                    if final_imputation_metrics_eval:
                        df_final_imputation_metrics = pd.DataFrame(final_imputation_metrics_eval)
                        print("\nMétricas de Calidad de Imputación Final - Resumen por Columna:")
                        print(df_final_imputation_metrics.to_string())

                        if all_true_final_eval and all_imputed_final_eval:
                            global_rmse_final_imputation = np.sqrt(mean_squared_error(all_true_final_eval, all_imputed_final_eval))
                            global_mae_final_imputation = mean_absolute_error(all_true_final_eval, all_imputed_final_eval)
                            print(f"\nRMSE Global (Imputación Final): {global_rmse_final_imputation:.3f} (sobre {len(all_true_final_eval)} puntos)")
                            print(f"MAE Global (Imputación Final): {global_mae_final_imputation:.3f} (sobre {len(all_true_final_eval)} puntos)")
                            
                            # Gráfico de Dispersión
                            plt.figure(figsize=(8, 8))
                            sample_size_final_eval = min(len(all_true_final_eval), 2000)
                            if sample_size_final_eval > 0:
                                indices_muestreados_final = np.random.choice(len(all_true_final_eval), size=sample_size_final_eval, replace=False)
                                true_sampled_final = np.array(all_true_final_eval)[indices_muestreados_final]
                                imputed_sampled_final = np.array(all_imputed_final_eval)[indices_muestreados_final]
                                
                                plt.scatter(true_sampled_final, imputed_sampled_final, alpha=0.5, s=10, label='Corrected Points', c='purple')
                                min_val_plot_final = np.nanmin(np.concatenate([true_sampled_final, imputed_sampled_final])) # Usar nanmin/nanmax
                                max_val_plot_final = np.nanmax(np.concatenate([true_sampled_final, imputed_sampled_final]))
                                if pd.notna(min_val_plot_final) and pd.notna(max_val_plot_final) and min_val_plot_final < max_val_plot_final:
                                    plt.plot([min_val_plot_final, max_val_plot_final], [min_val_plot_final, max_val_plot_final], 'r--', lw=2, label='Ideal')
                                plt.xlabel("True Values")
                                plt.ylabel("Corrected Values")
                                plt.legend()
                                plt.grid(True)
                                plt.tight_layout()
                                plt.show()
                            else:
                                print("\nNo hay suficientes datos válidos para graficar la imputación final.")
                        else:
                            print("\nNo hay suficientes datos globales para calcular RMSE/MAE para la imputación final.")
                    else:
                        print("\nNo se evaluaron puntos en la imputación final (quizás no había NaNs que coincidieran con valores originales para comparar).")
                else:
                    print("\nNo se pudo realizar la evaluación de la imputación final: 'df_original_eval_actual' no disponible o no se identificaron NaNs para evaluar.")
            else:
                print("\nNo hay columnas con NaNs restantes que requieran imputación en este paso final.")
else:
    print("IterativeImputer ('iterative_imputer_faltantes') no está disponible o no fue definido.")

print(f"\n===============================================================================")
print(f"FIN DE FASE DE IMPUTACIÓN FINAL Y EVALUACIÓN")
print(f"===============================================================================")

## 6. Umbrales adaptativos por sensor (v3 — M4)
Calculamos el coeficiente de variación (CV) de cada sensor. Sensores más estables (bajo CV) reciben umbrales más estrictos para la corrección.

In [ ]:
# v3 — M4: Umbrales adaptativos de corrección por sensor
# Factor = f(coeficiente de variación): más estricto para sensores estables
if 'df_original' in locals() and 'columnas_objetivo_existentes' in locals():
    _sensores_cv = [c for c in columnas_objetivo_existentes if c in df_original.columns
                    and pd.api.types.is_numeric_dtype(df_original[c])]
    factores_umbral = {}
    for col in _sensores_cv:
        _mean = df_original[col].mean()
        _std  = df_original[col].std()
        cv = _std / (abs(_mean) + 1e-6)
        if   cv < 0.10: factores_umbral[col] = 1.5   # sensor muy estable → más exigente
        elif cv < 0.50: factores_umbral[col] = 2.0   # variabilidad media
        else:           factores_umbral[col] = 2.5   # sensor muy variable
    print("Factores de umbral adaptativos (M4):")
    for col, f in factores_umbral.items():
        print(f"  {col:<10} factor={f}")
else:
    # Fallback: factor uniforme si no hay df_original
    factores_umbral = {}
    print("Advertencia: factores_umbral inicializados vacíos — se usará factor 2.0 por defecto.")
# ─── Variables requeridas por celdas 19-25 ───────────────────────────────────
# std devs de sensores para umbrales adaptativos
std_devs_originales = {}
_df_ref_std = df_original if 'df_original' in locals() else df_para_corregir
for _col_std in columnas_objetivo_existentes:
    if _col_std in _df_ref_std.columns:
        std_devs_originales[_col_std] = _df_ref_std[_col_std].std()

# Rangos válidos de sensores desde config
rangos_validos_sensores = RANGOS_FISICOS
print(f"std_devs_originales y rangos_validos_sensores definidos ({len(std_devs_originales)} sensores).")


## 7. Corrección de Ruido
Interpolación local: reemplazamos el valor ruidoso por la media del punto anterior y siguiente.

In [ ]:
# --- INICIO DEL PROCESO DE CORRECCIÓN DE RUIDO (MÉTODO PERSONALIZADO) ---

# Definir el nombre de la clase para "Ruido" según el modelo M2
nombre_clase_ruido_config = "Ruido" 
# v3 — M4: factor adaptativo por sensor (definido en celda anterior)
factor_umbral_diferencia = FACTOR_UMBRAL_CORRECCION

# Verificar que las variables necesarias existen
if 'df_stuck_hybrid_corrected' in locals() or 'df_stuck_hybrid_corrected' in globals():

    df_temp_ruido = locals().get('df_stuck_hybrid_corrected', globals().get('df_stuck_hybrid_corrected'))
    if 'pred_tipo_anomalia' in df_temp_ruido.columns:
        serie_predicciones_hybrid = df_temp_ruido['pred_tipo_anomalia']
        print("\n✅ Columna 'pred_tipo_anomalia' encontrada. Se filtrarán solo los predecidos como Ruido.")
    elif 'serie_predicciones_hybrid' not in locals() and 'serie_predicciones_hybrid' not in globals():
        # Intentamos reconstruir a partir del modelo
        y_pred = locals().get('y_pred_model2_actual', globals().get('y_pred_model2_actual', locals().get('y_pred_model2', globals().get('y_pred_model2'))))
        le_model = locals().get('label_encoder_model2_actual', globals().get('label_encoder_model2_actual', locals().get('label_encoder_model2', globals().get('label_encoder_model2'))))
        X_test = locals().get('X_model2_test_actual', globals().get('X_model2_test_actual', locals().get('X_model2_test', globals().get('X_model2_test'))))
        
        if y_pred is not None and le_model is not None and X_test is not None:
            try:
                serie_pred_temp = pd.Series(le_model.inverse_transform(y_pred), index=X_test.index)
                serie_predicciones_hybrid = pd.Series(dtype='object', index=df_temp_ruido.index)
                common_idx = df_temp_ruido.index.intersection(serie_pred_temp.index)
                serie_predicciones_hybrid.loc[common_idx] = serie_pred_temp.loc[common_idx]
                print("\n✅ Se usaron las variables del modelo para generar las predicciones de Ruido.")
            except Exception:
                serie_predicciones_hybrid = pd.Series(dtype='object')
        else:
            serie_predicciones_hybrid = pd.Series(dtype='object')

    if 'serie_predicciones_hybrid' in locals() or 'serie_predicciones_hybrid' in globals():
        if 'std_devs_originales' in locals() or 'std_devs_originales' in globals():
            if 'columnas_objetivo_existentes' in locals() or 'columnas_objetivo_existentes' in globals():

                print("\n===============================================================================")
                print(f"INICIANDO FASE DE CORRECCIÓN DE '{nombre_clase_ruido_config}' CON MÉTODO PERSONALIZADO")
                print(f"Factor umbral para diferencia (x std_dev): {factor_umbral_diferencia}")
                print("===============================================================================")

                df_ruido_corregido_custom = df_stuck_hybrid_corrected.copy()
                
                columnas_para_corregir_ruido_custom = [
                    col for col in columnas_objetivo_existentes if col in df_ruido_corregido_custom.columns
                ]
                
                if not columnas_para_corregir_ruido_custom:
                    print("Advertencia: No hay columnas objetivo válidas para la corrección de ruido personalizada. No se realizarán correcciones de ruido.")
                else:
                    print(f"Columnas objetivo para corrección de ruido: {columnas_para_corregir_ruido_custom}")

                    puntos_corregidos_count = {}
                    indices_ruidosos_para_eval_custom = [] 

                    print(f"\n--- Identificando y Corrigiendo '{nombre_clase_ruido_config}' con lógica personalizada ---")
                    
                    lista_puntos_ruidosos_identificados = []
                    print("Fase 1: Identificando todos los puntos ruidosos...")
                    for col_idx_check in columnas_para_corregir_ruido_custom:
                        if col_idx_check not in df_ruido_corregido_custom.columns:
                            continue
                        
                        temp_serie_pred_alineada_custom = pd.Series(dtype=object)
                        if not isinstance(df_ruido_corregido_custom.index, pd.Index) or not isinstance(serie_predicciones_hybrid.index, pd.Index):
                            # print(f"Advertencia (Alineación Ruido Custom): Índices no válidos en df_ruido_corregido_custom o serie_predicciones_hybrid para '{col_idx_check}'.")
                            continue

                        if isinstance(df_ruido_corregido_custom.index, pd.DatetimeIndex) and \
                            isinstance(serie_predicciones_hybrid.index, pd.DatetimeIndex):
                            if df_ruido_corregido_custom.index.equals(serie_predicciones_hybrid.index):
                                temp_serie_pred_alineada_custom = serie_predicciones_hybrid
                            else:
                                common_idx_pred_custom = df_ruido_corregido_custom.index.intersection(serie_predicciones_hybrid.index)
                                if not common_idx_pred_custom.empty:
                                    temp_serie_pred_alineada_custom = serie_predicciones_hybrid.loc[common_idx_pred_custom].reindex(df_ruido_corregido_custom.index)
                                else:
                                    continue
                        elif df_ruido_corregido_custom.index.equals(serie_predicciones_hybrid.index):
                            temp_serie_pred_alineada_custom = serie_predicciones_hybrid
                        else: 
                            try:
                                common_idx_fallback_custom = df_ruido_corregido_custom.index.intersection(serie_predicciones_hybrid.index)
                                if not common_idx_fallback_custom.empty:
                                    temp_serie_pred_alineada_custom = serie_predicciones_hybrid.loc[common_idx_fallback_custom].reindex(df_ruido_corregido_custom.index)
                                else:
                                    continue
                            except Exception: # Ignorar errores de alineación fallback silenciosamente por ahora
                                continue
                        
                        if temp_serie_pred_alineada_custom.empty or temp_serie_pred_alineada_custom.isnull().all():
                            continue

                        indices_ruido_col = temp_serie_pred_alineada_custom[temp_serie_pred_alineada_custom == nombre_clase_ruido_config].index
                        for idx_ruido_item in indices_ruido_col:
                            if idx_ruido_item in df_ruido_corregido_custom.index: # Asegurar que el índice es válido para el DF que se está corrigiendo
                                lista_puntos_ruidosos_identificados.append((col_idx_check, idx_ruido_item))
                                indices_ruidosos_para_eval_custom.append((col_idx_check, idx_ruido_item)) 
                    
                    print(f"Fase 1: {len(lista_puntos_ruidosos_identificados)} puntos ruidosos identificados en total.")
                    
                    # --- Preparar DataFrame fuente para vecinos con índice único ---
                    df_source_for_neighbors = df_stuck_hybrid_corrected.copy()
                    if df_source_for_neighbors.index.has_duplicates:
                        print("Advertencia (Ruido Custom - Preparación Vecinos): 'df_stuck_hybrid_corrected' (usado para obtener vecinos) tiene índices duplicados. Se tomará la primera aparición.")
                        df_source_for_neighbors = df_source_for_neighbors[~df_source_for_neighbors.index.duplicated(keep='first')]
                    
                    print("Fase 2: Aplicando corrección a los puntos identificados...")
                    for col_actual, idx_ruido in lista_puntos_ruidosos_identificados:
                        puntos_corregidos_count[col_actual] = puntos_corregidos_count.get(col_actual, 0) + 1
                        
                        i_pos = -1 
                        try:
                            # Usar df_source_for_neighbors que tiene índice (esperadamente) único
                            loc_result = df_source_for_neighbors.index.get_loc(idx_ruido)
                            
                            if isinstance(loc_result, int):
                                i_pos = loc_result
                            elif isinstance(loc_result, slice):
                                i_pos = loc_result.start
                                if i_pos is None: i_pos = -1
                            elif isinstance(loc_result, np.ndarray): 
                                if loc_result.dtype == bool: positions = np.where(loc_result)[0]
                                else: positions = loc_result
                                if len(positions) > 0: i_pos = positions[0] 
                                else: i_pos = -1
                                    
                            if i_pos == -1: # Si no se pudo determinar una posición única válida
                                df_ruido_corregido_custom.loc[idx_ruido, col_actual] = np.nan # Marcar como NaN
                                continue
                        except KeyError: # Si idx_ruido (de la identificación) no está en df_source_for_neighbors (después de de-duplicar)
                            # print(f"Advertencia (Ruido Custom GetLoc): Índice {idx_ruido} no encontrado en la fuente de vecinos de-duplicada. Saltando.")
                            df_ruido_corregido_custom.loc[idx_ruido, col_actual] = np.nan # Marcar como NaN
                            continue
                        except Exception as e_getloc:
                            print(f"Error inesperado en get_loc para índice {idx_ruido}, columna {col_actual}: {e_getloc}. Saltando.")
                            df_ruido_corregido_custom.loc[idx_ruido, col_actual] = np.nan # Marcar como NaN
                            continue

                        val_prev_original, val_next_original = np.nan, np.nan
                        prev_val_exists, next_val_exists = False, False

                        # Obtener vecinos de df_source_for_neighbors
                        if 0 <= i_pos < len(df_source_for_neighbors.index):
                            if i_pos > 0:
                                idx_prev_ts = df_source_for_neighbors.index[i_pos - 1]
                                # .loc en df_source_for_neighbors (con índice único) debería dar escalar
                                val_prev_original = df_source_for_neighbors.loc[idx_prev_ts, col_actual] 
                                if pd.notna(val_prev_original):
                                    prev_val_exists = True
                            
                            if i_pos < len(df_source_for_neighbors.index) - 1:
                                idx_next_ts = df_source_for_neighbors.index[i_pos + 1]
                                # .loc en df_source_for_neighbors (con índice único) debería dar escalar
                                val_next_original = df_source_for_neighbors.loc[idx_next_ts, col_actual]
                                if pd.notna(val_next_original): 
                                    next_val_exists = True
                        else:
                            df_ruido_corregido_custom.loc[idx_ruido, col_actual] = np.nan
                            continue

                        corrected_value = np.nan 

                        if prev_val_exists and next_val_exists:
                            diff = abs(val_next_original - val_prev_original)
                            std_dev_col = std_devs_originales.get(col_actual, 0) 
                            threshold = factor_umbral_diferencia * std_dev_col if std_dev_col > 0 else float('inf')

                            if diff > threshold :
                                corrected_value = val_prev_original
                            else:
                                corrected_value = (val_prev_original + val_next_original) / 2
                        elif prev_val_exists:
                            corrected_value = val_prev_original
                        elif next_val_exists:
                            corrected_value = val_next_original
                        
                        df_ruido_corregido_custom.loc[idx_ruido, col_actual] = corrected_value

                    for col_rep, count_rep in puntos_corregidos_count.items():
                        if count_rep > 0:
                            print(f"   En columna '{col_rep}': {count_rep} puntos '{nombre_clase_ruido_config}' procesados con método personalizado.")
                    
                    # --- Evaluación de la Corrección de Ruido Personalizada ---
                    df_original_ref_eval_custom = locals().get('df_original_eval_actual', globals().get('df_original_eval_actual'))
                    if df_original_ref_eval_custom is None:
                        df_original_ref_eval_custom = locals().get('df_original', globals().get('df_original'))

                    if df_original_ref_eval_custom is not None and indices_ruidosos_para_eval_custom:
                        
                        print("\n--- Evaluación Detallada de Corrección de Ruido (Método Personalizado) ---")
                        
                        all_true_ruido_custom_eval = []
                        all_imputed_ruido_custom_eval = []
                        ruido_custom_metrics_eval = []

                        df_original_ref_eval_custom = df_original_ref_eval_custom.copy()
                        df_ruido_corregido_eval_custom = df_ruido_corregido_custom.copy() 

                        if df_original_ref_eval_custom.index.has_duplicates:
                            df_original_ref_eval_custom = df_original_ref_eval_custom[~df_original_ref_eval_custom.index.duplicated(keep='first')]
                        
                        if df_ruido_corregido_eval_custom.index.has_duplicates:
                            df_ruido_corregido_eval_custom = df_ruido_corregido_eval_custom[~df_ruido_corregido_eval_custom.index.duplicated(keep='first')]

                        eval_data_by_col_custom = defaultdict(lambda: {'true': [], 'imputed': [], 'indices': []})

                        for col_eval_c, idx_eval_c in indices_ruidosos_para_eval_custom:
                            if not (idx_eval_c in df_original_ref_eval_custom.index and \
                                    idx_eval_c in df_ruido_corregido_eval_custom.index and \
                                    col_eval_c in df_original_ref_eval_custom.columns and \
                                    col_eval_c in df_ruido_corregido_eval_custom.columns):
                                continue
                            
                            true_val_c = df_original_ref_eval_custom.loc[idx_eval_c, col_eval_c]
                            imputed_val_c = df_ruido_corregido_eval_custom.loc[idx_eval_c, col_eval_c]

                            if pd.notna(true_val_c) and pd.notna(imputed_val_c): 
                                eval_data_by_col_custom[col_eval_c]['true'].append(true_val_c)
                                eval_data_by_col_custom[col_eval_c]['imputed'].append(imputed_val_c)
                                eval_data_by_col_custom[col_eval_c]['indices'].append(idx_eval_c)
                        
                        for col_eval_metric_c, data_c in eval_data_by_col_custom.items():
                            if not data_c['true']: continue
                            true_vals_series_c = pd.Series(data_c['true'])
                            imputed_vals_series_c = pd.Series(data_c['imputed'])
                            
                            all_true_ruido_custom_eval.extend(true_vals_series_c.tolist())
                            all_imputed_ruido_custom_eval.extend(imputed_vals_series_c.tolist())
                            
                            rmse_col_c = np.sqrt(mean_squared_error(true_vals_series_c, imputed_vals_series_c))
                            mae_col_c = mean_absolute_error(true_vals_series_c, imputed_vals_series_c)
                            
                            ruido_custom_metrics_eval.append({
                                'Columna': col_eval_metric_c,
                                'Num_Puntos_Corregidos': len(true_vals_series_c),
                                'RMSE_Correccion_Custom': rmse_col_c,
                                'MAE_Correccion_Custom': mae_col_c,
                                'Media_Real_Original': true_vals_series_c.mean(),
                                'Media_Imputada_Custom': imputed_vals_series_c.mean()
                            })

                        if ruido_custom_metrics_eval:
                            df_ruido_custom_eval_metrics = pd.DataFrame(ruido_custom_metrics_eval)
                            print("\nMétricas de Calidad de Corrección de Ruido (Método Personalizado) - Resumen por Columna:")
                            print(df_ruido_custom_eval_metrics.to_string())

                            if all_true_ruido_custom_eval and all_imputed_ruido_custom_eval:
                                global_rmse_custom = np.sqrt(mean_squared_error(all_true_ruido_custom_eval, all_imputed_ruido_custom_eval))
                                global_mae_custom = mean_absolute_error(all_true_ruido_custom_eval, all_imputed_ruido_custom_eval)
                                print(f"\nRMSE Global (Corrección Ruido Custom): {global_rmse_custom:.3f} (sobre {len(all_true_ruido_custom_eval)} puntos)")
                                print(f"MAE Global (Corrección Ruido Custom): {global_mae_custom:.3f} (sobre {len(all_true_ruido_custom_eval)} puntos)")
                                
                                plt.figure(figsize=(8, 8))
                                sample_size_custom = min(len(all_true_ruido_custom_eval), 2000)
                                if sample_size_custom > 0:
                                    indices_smp_custom = np.random.choice(len(all_true_ruido_custom_eval), size=sample_size_custom, replace=False)
                                    true_smp_c = np.array(all_true_ruido_custom_eval)[indices_smp_custom]
                                    imputed_smp_c = np.array(all_imputed_ruido_custom_eval)[indices_smp_custom]
                                    
                                    plt.scatter(true_smp_c, imputed_smp_c, alpha=0.5, s=10, label='Corrected Points', c='green')
                                    min_val_plot_c = np.nanmin(np.concatenate([true_smp_c, imputed_smp_c]))
                                    max_val_plot_c = np.nanmax(np.concatenate([true_smp_c, imputed_smp_c]))
                                    if pd.notna(min_val_plot_c) and pd.notna(max_val_plot_c) and min_val_plot_c < max_val_plot_c:
                                        plt.plot([min_val_plot_c, max_val_plot_c], [min_val_plot_c, max_val_plot_c], 'r--', lw=2, label='Ideal')
                                    plt.xlabel("True Values")
                                    plt.ylabel(f"Corrected Values")
                                    plt.legend()
                                    plt.grid(True)
                                    plt.tight_layout()
                                    plt.show()
                        else: 
                            print("\nNo se generaron métricas para la corrección de ruido personalizada.")
                    else: 
                        print("\nNo se pudo realizar la evaluación de corrección de ruido personalizada (falta df_original o no hay puntos para evaluar).")

                    print("\n===============================================================================")
                    print(f"FIN DE LA FASE DE CORRECCIÓN DE '{nombre_clase_ruido_config}' (MÉTODO PERSONALIZADO)")
                    print("===============================================================================")
                    print("\nEl DataFrame con las correcciones (ruido con método custom) está en: df_ruido_corregido_custom")
            else:
                print("Error: La variable 'columnas_objetivo_existentes' no está definida o está vacía. No se puede proceder con la corrección de ruido.")
        else: 
            print("Error: La variable 'std_devs_originales' no está disponible. No se puede proceder con la corrección de ruido personalizada.")
    else: 
        print("Error: La variable 'serie_predicciones_hybrid' no está disponible. No se puede proceder con la corrección de ruido personalizada.")
else:
    print("Error: El DataFrame 'df_stuck_hybrid_corrected' no está disponible. Asegúrate de que los pasos anteriores se hayan ejecutado correctamente.")

## 8. Corrección de Valores Fuera de Rango

In [ ]:
# Factor para el umbral de diferencia entre anterior y siguiente para esta corrección
# v3 — M4: factor adaptativo por sensor
factor_umbral_diferencia_oor = FACTOR_UMBRAL_CORRECCION

nombre_clase_oor_config = "Valores Fuera de Rango"

# Verificar que las variables necesarias existen
if 'df_ruido_corregido_custom' in locals() or 'df_ruido_corregido_custom' in globals():

    df_temp_oor = locals().get('df_ruido_corregido_custom', globals().get('df_ruido_corregido_custom'))
    if 'pred_tipo_anomalia' in df_temp_oor.columns:
        serie_predicciones_hybrid = df_temp_oor['pred_tipo_anomalia']
        print("\n✅ Columna 'pred_tipo_anomalia' encontrada. Se filtrarán solo los predecidos como Valores Fuera de Rango.")
    elif 'serie_predicciones_hybrid' not in locals() and 'serie_predicciones_hybrid' not in globals():
        # Intentamos reconstruir a partir del modelo
        y_pred = locals().get('y_pred_model2_actual', globals().get('y_pred_model2_actual', locals().get('y_pred_model2', globals().get('y_pred_model2'))))
        le_model = locals().get('label_encoder_model2_actual', globals().get('label_encoder_model2_actual', locals().get('label_encoder_model2', globals().get('label_encoder_model2'))))
        X_test = locals().get('X_model2_test_actual', globals().get('X_model2_test_actual', locals().get('X_model2_test', globals().get('X_model2_test'))))
        
        if y_pred is not None and le_model is not None and X_test is not None:
            try:
                serie_pred_temp = pd.Series(le_model.inverse_transform(y_pred), index=X_test.index)
                serie_predicciones_hybrid = pd.Series(dtype='object', index=df_temp_oor.index)
                common_idx = df_temp_oor.index.intersection(serie_pred_temp.index)
                serie_predicciones_hybrid.loc[common_idx] = serie_pred_temp.loc[common_idx]
                print("\n✅ Se usaron las variables del modelo para generar las predicciones de Fuera de Rango.")
            except Exception:
                serie_predicciones_hybrid = pd.Series(dtype='object')
        else:
            serie_predicciones_hybrid = pd.Series(dtype='object')
    else:
        serie_predicciones_hybrid = locals().get('serie_predicciones_hybrid', globals().get('serie_predicciones_hybrid'))

    if 'std_devs_originales' in locals() or 'std_devs_originales' in globals():
        if 'columnas_objetivo_existentes' in locals() or 'columnas_objetivo_existentes' in globals():
            if 'rangos_validos_sensores' in locals() or 'rangos_validos_sensores' in globals():

                print("\n===============================================================================")
                print(f"INICIANDO FASE DE CORRECCIÓN DE VALORES FUERA DE RANGO CON MÉTODO PERSONALIZADO")
                print(f"Factor umbral para diferencia de vecinos (x std_dev): {factor_umbral_diferencia_oor}")
                print("===============================================================================")

                df_oor_corregido = df_ruido_corregido_custom.copy()
                
                # Columnas donde se aplicará la corrección
                # (aquellas presentes en rangos_validos_sensores, columnas_objetivo_existentes y el df actual)
                columnas_para_corregir_oor = [
                    col for col in rangos_validos_sensores.keys() 
                    if col in df_oor_corregido.columns and col in columnas_objetivo_existentes
                ]

                if not columnas_para_corregir_oor:
                    print("Advertencia: No hay columnas válidas (con rangos definidos y presentes en el DF) para la corrección de valores fuera de rango.")
                else:
                    print(f"Columnas objetivo para corrección de fuera de rango: {columnas_para_corregir_oor}")

                    puntos_oor_corregidos_count = {}
                    indices_fuera_rango_para_eval = [] 

                    print(f"\n--- Identificando y Corrigiendo Valores Fuera de Rango ---")
                    
                    lista_puntos_fuera_rango_identificados = []
                    print("Fase 1: Identificando todos los puntos fuera de rango...")
                    for col_check in columnas_para_corregir_oor:
                        # Obtener límites del diccionario provisto por el usuario
                        if col_check not in rangos_validos_sensores: # Seguridad adicional
                            # print(f"Advertencia: Rangos no definidos para {col_check} en 'rangos_validos_sensores'. Saltando.")
                            continue
                        
                        rango_col = rangos_validos_sensores[col_check]
                        min_limit = rango_col.get('min')
                        max_limit = rango_col.get('max')

                        if min_limit is None or max_limit is None:
                            # print(f"Advertencia: Límites 'min' o 'max' no especificados para {col_check} en 'rangos_validos_sensores'. Saltando.")
                            continue

                        # Alineación de predicciones para cruzar datos
                        temp_serie_pred_alineada_oor = pd.Series(dtype=object)
                        if isinstance(df_oor_corregido.index, pd.Index) and isinstance(serie_predicciones_hybrid.index, pd.Index) and not serie_predicciones_hybrid.empty:
                            if df_oor_corregido.index.equals(serie_predicciones_hybrid.index):
                                temp_serie_pred_alineada_oor = serie_predicciones_hybrid
                            else:
                                common_idx_oor = df_oor_corregido.index.intersection(serie_predicciones_hybrid.index)
                                if not common_idx_oor.empty:
                                    temp_serie_pred_alineada_oor = serie_predicciones_hybrid.loc[common_idx_oor].reindex(df_oor_corregido.index)
                        
                        # Asegurarse de que la columna es numérica
                        if pd.api.types.is_numeric_dtype(df_oor_corregido[col_check]):
                            # Identificar índices donde el valor está fuera de rango
                            # pd.to_numeric para asegurar que la comparación sea numérica y manejar errores de conversión
                            col_data_numeric = pd.to_numeric(df_oor_corregido[col_check], errors='coerce')
                            out_of_range_mask = (col_data_numeric < min_limit) | (col_data_numeric > max_limit)

                            # Intersección de la regla estricta (diccionario) con la predicción (modelo)
                            if not temp_serie_pred_alineada_oor.empty:
                                mask_pred_oor = (temp_serie_pred_alineada_oor == nombre_clase_oor_config)
                                mascara_final = out_of_range_mask & col_data_numeric.notna() & mask_pred_oor
                            else:
                                # Fallback si no hay predicciones: usar solo el diccionario como heurística
                                mascara_final = out_of_range_mask & col_data_numeric.notna()

                            indices_oor_col = df_oor_corregido[mascara_final].index # Solo considerar no-NaNs fuera de rango
                            
                            for idx_oor_item in indices_oor_col:
                                lista_puntos_fuera_rango_identificados.append((col_check, idx_oor_item))
                                indices_fuera_rango_para_eval.append((col_check, idx_oor_item))
                        else:
                            # print(f"Advertencia: Columna {col_check} no es numérica, se omitirá para la detección de fuera de rango.")
                            pass


                    print(f"Fase 1: {len(lista_puntos_fuera_rango_identificados)} puntos fuera de rango identificados en total.")
                    
                    df_source_for_neighbors_oor = df_ruido_corregido_custom.copy() # Vecinos se toman del DF *antes* de esta corrección OOR
                    if df_source_for_neighbors_oor.index.has_duplicates:
                        print("Advertencia (OOR Custom - Preparación Vecinos): DataFrame fuente para vecinos tiene índices duplicados. Se tomará la primera aparición.")
                        df_source_for_neighbors_oor = df_source_for_neighbors_oor[~df_source_for_neighbors_oor.index.duplicated(keep='first')]
                    
                    print("Fase 2: Aplicando corrección a los puntos fuera de rango identificados...")
                    for col_actual, idx_oor in lista_puntos_fuera_rango_identificados:
                        puntos_oor_corregidos_count[col_actual] = puntos_oor_corregidos_count.get(col_actual, 0) + 1
                        
                        i_pos = -1 
                        try:
                            loc_result = df_source_for_neighbors_oor.index.get_loc(idx_oor)
                            
                            if isinstance(loc_result, int): i_pos = loc_result
                            elif isinstance(loc_result, slice):
                                i_pos = loc_result.start
                                if i_pos is None: i_pos = -1
                            elif isinstance(loc_result, np.ndarray): 
                                if loc_result.dtype == bool: positions = np.where(loc_result)[0]
                                else: positions = loc_result
                                if len(positions) > 0: i_pos = positions[0] 
                                else: i_pos = -1
                                    
                            if i_pos == -1:
                                df_oor_corregido.loc[idx_oor, col_actual] = np.nan
                                continue
                        except KeyError:
                            df_oor_corregido.loc[idx_oor, col_actual] = np.nan
                            continue
                        except Exception as e_getloc_oor:
                            print(f"Error inesperado en get_loc (OOR) para índice {idx_oor}, columna {col_actual}: {e_getloc_oor}. Saltando.")
                            df_oor_corregido.loc[idx_oor, col_actual] = np.nan
                            continue

                        val_prev_original, val_next_original = np.nan, np.nan
                        prev_val_exists, next_val_exists = False, False

                        if 0 <= i_pos < len(df_source_for_neighbors_oor.index):
                            if i_pos > 0:
                                idx_prev_ts = df_source_for_neighbors_oor.index[i_pos - 1]
                                val_prev_original = df_source_for_neighbors_oor.loc[idx_prev_ts, col_actual]
                                if pd.notna(val_prev_original): prev_val_exists = True
                            
                            if i_pos < len(df_source_for_neighbors_oor.index) - 1:
                                idx_next_ts = df_source_for_neighbors_oor.index[i_pos + 1]
                                val_next_original = df_source_for_neighbors_oor.loc[idx_next_ts, col_actual]
                                if pd.notna(val_next_original): next_val_exists = True
                        else:
                            df_oor_corregido.loc[idx_oor, col_actual] = np.nan
                            continue

                        corrected_value = np.nan 
                        if prev_val_exists and next_val_exists:
                            diff = abs(val_next_original - val_prev_original)
                            std_dev_col = std_devs_originales.get(col_actual, 0) 
                            threshold = factor_umbral_diferencia_oor * std_dev_col if std_dev_col > 0 else float('inf')

                            if diff > threshold : corrected_value = val_prev_original
                            else: corrected_value = (val_prev_original + val_next_original) / 2
                        elif prev_val_exists: corrected_value = val_prev_original
                        elif next_val_exists: corrected_value = val_next_original
                        
                        df_oor_corregido.loc[idx_oor, col_actual] = corrected_value

                    for col_rep, count_rep in puntos_oor_corregidos_count.items():
                        if count_rep > 0:
                            print(f"   En columna '{col_rep}': {count_rep} puntos Fuera de Rango procesados con método personalizado.")
                    
                    # --- Evaluación de la Corrección de Fuera de Rango ---
                    df_original_ref_eval_oor = locals().get('df_original_eval_actual', globals().get('df_original_eval_actual'))
                    if df_original_ref_eval_oor is None:
                        df_original_ref_eval_oor = locals().get('df_original', globals().get('df_original'))

                    if df_original_ref_eval_oor is not None and indices_fuera_rango_para_eval:
                        
                        print("\n--- Evaluación Detallada de Corrección de Valores Fuera de Rango (Método Personalizado) ---")
                        all_true_oor_eval = []
                        all_imputed_oor_eval = []
                        oor_custom_metrics_eval = []

                        df_original_ref_eval_oor = df_original_ref_eval_oor.copy()
                        df_oor_corregido_eval = df_oor_corregido.copy()

                        if df_original_ref_eval_oor.index.has_duplicates:
                            df_original_ref_eval_oor = df_original_ref_eval_oor[~df_original_ref_eval_oor.index.duplicated(keep='first')]
                        if df_oor_corregido_eval.index.has_duplicates:
                            df_oor_corregido_eval = df_oor_corregido_eval[~df_oor_corregido_eval.index.duplicated(keep='first')]

                        eval_data_by_col_oor = defaultdict(lambda: {'true': [], 'imputed': [], 'indices': []})

                        for col_eval_oor, idx_eval_oor in indices_fuera_rango_para_eval:
                            if not (idx_eval_oor in df_original_ref_eval_oor.index and \
                                    idx_eval_oor in df_oor_corregido_eval.index and \
                                    col_eval_oor in df_original_ref_eval_oor.columns and \
                                    col_eval_oor in df_oor_corregido_eval.columns):
                                continue
                            
                            true_val_oor = df_original_ref_eval_oor.loc[idx_eval_oor, col_eval_oor]
                            imputed_val_oor = df_oor_corregido_eval.loc[idx_eval_oor, col_eval_oor]

                            if pd.notna(true_val_oor) and pd.notna(imputed_val_oor): 
                                eval_data_by_col_oor[col_eval_oor]['true'].append(true_val_oor)
                                eval_data_by_col_oor[col_eval_oor]['imputed'].append(imputed_val_oor)
                                eval_data_by_col_oor[col_eval_oor]['indices'].append(idx_eval_oor)
                        
                        for col_eval_metric_oor, data_oor in eval_data_by_col_oor.items():
                            if not data_oor['true']: continue
                            true_vals_series_oor = pd.Series(data_oor['true'])
                            imputed_vals_series_oor = pd.Series(data_oor['imputed'])
                            
                            all_true_oor_eval.extend(true_vals_series_oor.tolist())
                            all_imputed_oor_eval.extend(imputed_vals_series_oor.tolist())
                            
                            rmse_col_oor = np.sqrt(mean_squared_error(true_vals_series_oor, imputed_vals_series_oor))
                            mae_col_oor = mean_absolute_error(true_vals_series_oor, imputed_vals_series_oor)
                            
                            oor_custom_metrics_eval.append({
                                'Columna': col_eval_metric_oor,
                                'Num_Puntos_Corregidos': len(true_vals_series_oor),
                                'RMSE_Correccion_OOR': rmse_col_oor,
                                'MAE_Correccion_OOR': mae_col_oor,
                                'Media_Real_Original': true_vals_series_oor.mean(),
                                'Media_Imputada_OOR': imputed_vals_series_oor.mean()
                            })

                        if oor_custom_metrics_eval:
                            df_oor_custom_eval_metrics = pd.DataFrame(oor_custom_metrics_eval)
                            print("\nMétricas de Calidad de Corrección Fuera de Rango (Método Personalizado) - Resumen por Columna:")
                            print(df_oor_custom_eval_metrics.to_string())

                            if all_true_oor_eval and all_imputed_oor_eval:
                                global_rmse_oor = np.sqrt(mean_squared_error(all_true_oor_eval, all_imputed_oor_eval))
                                global_mae_oor = mean_absolute_error(all_true_oor_eval, all_imputed_oor_eval)
                                print(f"\nRMSE Global (Corrección Fuera de Rango): {global_rmse_oor:.3f} (sobre {len(all_true_oor_eval)} puntos)")
                                print(f"MAE Global (Corrección Fuera de Rango): {global_mae_oor:.3f} (sobre {len(all_true_oor_eval)} puntos)")
                                
                                plt.figure(figsize=(8,8))
                                sample_size_oor = min(len(all_true_oor_eval), 2000)
                                if sample_size_oor > 0:
                                    indices_smp_oor = np.random.choice(len(all_true_oor_eval), size=sample_size_oor, replace=False)
                                    true_smp_oor = np.array(all_true_oor_eval)[indices_smp_oor]
                                    imputed_smp_oor = np.array(all_imputed_oor_eval)[indices_smp_oor]
                                    
                                    plt.scatter(true_smp_oor, imputed_smp_oor, alpha=0.5, s=10, label='Corrected Points', c='purple')
                                    min_val_plot_oor = np.nanmin(np.concatenate([true_smp_oor, imputed_smp_oor]))
                                    max_val_plot_oor = np.nanmax(np.concatenate([true_smp_oor, imputed_smp_oor]))
                                    if pd.notna(min_val_plot_oor) and pd.notna(max_val_plot_oor) and min_val_plot_oor < max_val_plot_oor:
                                        plt.plot([min_val_plot_oor, max_val_plot_oor], [min_val_plot_oor, max_val_plot_oor], 'r--', lw=2, label='Ideal')
                                    plt.xlabel("True Values")
                                    plt.ylabel("Corrected Values")
                                    plt.legend()
                                    plt.grid(True)
                                    plt.tight_layout()
                                    plt.show()
                        else: 
                            print("\nNo se generaron métricas para la corrección de valores fuera de rango.")
                    else: 
                        print("\nNo se pudo realizar la evaluación de corrección de valores fuera de rango.")

                    print("\n===============================================================================")
                    print(f"FIN DE LA FASE DE CORRECCIÓN DE VALORES FUERA DE RANGO (MÉTODO PERSONALIZADO)")
                    print("===============================================================================")
                    print("\nEl DataFrame con las correcciones (fuera de rango con método custom) está en: df_oor_corregido")
                
            else:
                print("Error: El diccionario 'rangos_validos_sensores' no está definido. No se puede proceder.")
        else: 
            print("Error: 'columnas_objetivo_existentes' no está disponible. No se puede proceder.")
    else: 
        print("Error: 'std_devs_originales' no está disponible. No se puede proceder.")
else: 
    print("Error: El DataFrame resultante de la corrección de ruido ('df_ruido_corregido_custom') no está disponible.")

## 9. Corrección de Desviación de Correlación

In [ ]:
nombre_clase_corr_dev_config = "Desviación de Correlación"

# Factor para el umbral de diferencia entre anterior y siguiente para esta corrección
# v3 — M4
factor_umbral_diferencia_corr_dev = FACTOR_UMBRAL_CORRECCION

# Verificar que las variables necesarias existen
# El DataFrame de entrada es df_oor_corregido (resultado de la corrección anterior)
if 'df_oor_corregido' in locals() or 'df_oor_corregido' in globals():

    df_temp_corr = locals().get('df_oor_corregido', globals().get('df_oor_corregido'))
    if 'pred_tipo_anomalia' in df_temp_corr.columns:
        serie_predicciones_hybrid = df_temp_corr['pred_tipo_anomalia']
        print(f"\n✅ Columna 'pred_tipo_anomalia' encontrada. Se filtrarán solo los predecidos como '{nombre_clase_corr_dev_config}'.")
    elif 'serie_predicciones_hybrid' not in locals() and 'serie_predicciones_hybrid' not in globals():
        # Intentamos reconstruir a partir del modelo
        y_pred = locals().get('y_pred_model2_actual', globals().get('y_pred_model2_actual', locals().get('y_pred_model2', globals().get('y_pred_model2'))))
        le_model = locals().get('label_encoder_model2_actual', globals().get('label_encoder_model2_actual', locals().get('label_encoder_model2', globals().get('label_encoder_model2'))))
        X_test = locals().get('X_model2_test_actual', globals().get('X_model2_test_actual', locals().get('X_model2_test', globals().get('X_model2_test'))))
        
        if y_pred is not None and le_model is not None and X_test is not None:
            try:
                serie_pred_temp = pd.Series(le_model.inverse_transform(y_pred), index=X_test.index)
                serie_predicciones_hybrid = pd.Series(dtype='object', index=df_temp_corr.index)
                common_idx = df_temp_corr.index.intersection(serie_pred_temp.index)
                serie_predicciones_hybrid.loc[common_idx] = serie_pred_temp.loc[common_idx]
                print(f"\n✅ Se usaron las variables del modelo para generar las predicciones de '{nombre_clase_corr_dev_config}'.")
            except Exception:
                serie_predicciones_hybrid = pd.Series(dtype='object')
        else:
            serie_predicciones_hybrid = pd.Series(dtype='object')
    else:
        serie_predicciones_hybrid = locals().get('serie_predicciones_hybrid', globals().get('serie_predicciones_hybrid'))
    
        if 'std_devs_originales' in locals() or 'std_devs_originales' in globals():
            if 'columnas_objetivo_existentes' in locals() or 'columnas_objetivo_existentes' in globals():
                if 'df_original_eval_actual' in locals() or 'df_original_eval_actual' in globals():

                    print("\n===============================================================================")
                    print(f"INICIANDO FASE DE CORRECCIÓN DE '{nombre_clase_corr_dev_config}' CON MÉTODO PERSONALIZADO")
                    print(f"Factor umbral para diferencia (x std_dev): {factor_umbral_diferencia_corr_dev}")
                    print("===============================================================================")

                    df_corr_dev_corregido_custom = df_oor_corregido.copy()
                    
                    columnas_para_corregir_corr_dev_custom = [
                        col for col in columnas_objetivo_existentes if col in df_corr_dev_corregido_custom.columns
                    ]
                    
                    if not columnas_para_corregir_corr_dev_custom:
                        print("Advertencia: No hay columnas objetivo válidas para la corrección de Desviación de Correlación personalizada. No se realizarán correcciones.")
                    else:
                        print(f"Columnas objetivo para corrección de Desviación de Correlación: {columnas_para_corregir_corr_dev_custom}")

                        puntos_corr_dev_corregidos_count = {}
                        indices_corr_dev_para_eval_custom = [] 

                        print(f"\n--- Identificando y Corrigiendo '{nombre_clase_corr_dev_config}' con lógica personalizada ---")
                        
                        lista_puntos_corr_dev_identificados = []
                        print("Fase 1: Identificando todos los puntos con Desviación de Correlación...")
                        for col_idx_check in columnas_para_corregir_corr_dev_custom:
                            if col_idx_check not in df_corr_dev_corregido_custom.columns:
                                continue
                            
                            # Alineación de la serie de predicciones con el DataFrame actual
                            temp_serie_pred_alineada_corr_dev = pd.Series(dtype=object)
                            current_df_index = df_corr_dev_corregido_custom.index
                            predictions_index = serie_predicciones_hybrid.index

                            if not isinstance(current_df_index, pd.Index) or not isinstance(predictions_index, pd.Index):
                                print(f"Advertencia (Alineación CorrDev Custom): Índices no válidos en df_corr_dev_corregido_custom o serie_predicciones_hybrid para '{col_idx_check}'.")
                                continue

                            if current_df_index.equals(predictions_index):
                                temp_serie_pred_alineada_corr_dev = serie_predicciones_hybrid
                            else:
                                common_idx_pred_corr_dev = current_df_index.intersection(predictions_index)
                                if not common_idx_pred_corr_dev.empty:
                                    temp_serie_pred_alineada_corr_dev = serie_predicciones_hybrid.loc[common_idx_pred_corr_dev].reindex(current_df_index)
                                else:
                                    # print(f"Advertencia (Alineación CorrDev Custom): No hay índices comunes para '{col_idx_check}'.")
                                    continue
                            
                            if temp_serie_pred_alineada_corr_dev.empty or temp_serie_pred_alineada_corr_dev.isnull().all():
                                # print(f"Advertencia (Alineación CorrDev Custom): Serie de predicciones alineada vacía o toda NaN para '{col_idx_check}'.")
                                continue

                            indices_corr_dev_col = temp_serie_pred_alineada_corr_dev[temp_serie_pred_alineada_corr_dev == nombre_clase_corr_dev_config].index
                            for idx_corr_dev_item in indices_corr_dev_col:
                                if idx_corr_dev_item in df_corr_dev_corregido_custom.index: 
                                    lista_puntos_corr_dev_identificados.append((col_idx_check, idx_corr_dev_item))
                                    indices_corr_dev_para_eval_custom.append((col_idx_check, idx_corr_dev_item)) 
                        
                        print(f"Fase 1: {len(lista_puntos_corr_dev_identificados)} puntos con '{nombre_clase_corr_dev_config}' identificados en total.")
                        
                        # DataFrame fuente para vecinos (estado antes de esta corrección específica)
                        df_source_for_neighbors_corr_dev = df_oor_corregido.copy()
                        if df_source_for_neighbors_corr_dev.index.has_duplicates:
                            print("Advertencia (CorrDev Custom - Preparación Vecinos): DataFrame fuente ('df_oor_corregido') para vecinos tiene duplicados. Se tomará la primera aparición.")
                            df_source_for_neighbors_corr_dev = df_source_for_neighbors_corr_dev[~df_source_for_neighbors_corr_dev.index.duplicated(keep='first')]
                        
                        print("Fase 2: Aplicando corrección a los puntos identificados...")
                        for col_actual, idx_corr_dev in lista_puntos_corr_dev_identificados:
                            puntos_corr_dev_corregidos_count[col_actual] = puntos_corr_dev_corregidos_count.get(col_actual, 0) + 1
                            
                            i_pos = -1 
                            try:
                                loc_result = df_source_for_neighbors_corr_dev.index.get_loc(idx_corr_dev)
                                
                                if isinstance(loc_result, int): i_pos = loc_result
                                elif isinstance(loc_result, slice):
                                    i_pos = loc_result.start
                                    if i_pos is None: i_pos = -1
                                elif isinstance(loc_result, np.ndarray): 
                                    if loc_result.dtype == bool: positions = np.where(loc_result)[0]
                                    else: positions = loc_result
                                    if len(positions) > 0: i_pos = positions[0] 
                                    else: i_pos = -1
                                        
                                if i_pos == -1:
                                    df_corr_dev_corregido_custom.loc[idx_corr_dev, col_actual] = np.nan
                                    continue
                            except KeyError:
                                df_corr_dev_corregido_custom.loc[idx_corr_dev, col_actual] = np.nan
                                continue
                            except Exception as e_getloc:
                                print(f"Error inesperado en get_loc (CorrDev) para índice {idx_corr_dev}, columna {col_actual}: {e_getloc}. Saltando.")
                                df_corr_dev_corregido_custom.loc[idx_corr_dev, col_actual] = np.nan
                                continue

                            val_prev_original, val_next_original = np.nan, np.nan
                            prev_val_exists, next_val_exists = False, False

                            if 0 <= i_pos < len(df_source_for_neighbors_corr_dev.index):
                                if i_pos > 0:
                                    idx_prev_ts = df_source_for_neighbors_corr_dev.index[i_pos - 1]
                                    val_prev_original = df_source_for_neighbors_corr_dev.loc[idx_prev_ts, col_actual] 
                                    if pd.notna(val_prev_original): prev_val_exists = True
                                
                                if i_pos < len(df_source_for_neighbors_corr_dev.index) - 1:
                                    idx_next_ts = df_source_for_neighbors_corr_dev.index[i_pos + 1]
                                    val_next_original = df_source_for_neighbors_corr_dev.loc[idx_next_ts, col_actual]
                                    if pd.notna(val_next_original): next_val_exists = True
                            else:
                                df_corr_dev_corregido_custom.loc[idx_corr_dev, col_actual] = np.nan
                                continue

                            corrected_value = np.nan 
                            if prev_val_exists and next_val_exists:
                                diff = abs(val_next_original - val_prev_original)
                                std_dev_col = std_devs_originales.get(col_actual, 0) 
                                threshold = factor_umbral_diferencia_corr_dev * std_dev_col if std_dev_col > 0 else float('inf')

                                if diff > threshold : 
                                    corrected_value = val_prev_original 
                                else: 
                                    corrected_value = (val_prev_original + val_next_original) / 2
                            elif prev_val_exists: 
                                corrected_value = val_prev_original
                            elif next_val_exists: 
                                corrected_value = val_next_original
                            
                            df_corr_dev_corregido_custom.loc[idx_corr_dev, col_actual] = corrected_value

                        for col_rep, count_rep in puntos_corr_dev_corregidos_count.items():
                            if count_rep > 0:
                                print(f"   En columna '{col_rep}': {count_rep} puntos '{nombre_clase_corr_dev_config}' procesados con método personalizado.")
                        
                        # --- Evaluación de la Corrección de Desviación de Correlación Personalizada ---
                        df_original_eval_actual_ref = locals().get('df_original_eval_actual', globals().get('df_original_eval_actual'))
                        if df_original_eval_actual_ref is None:
                            df_original_eval_actual_ref = locals().get('df_original', globals().get('df_original'))


                        if df_original_eval_actual_ref is not None and indices_corr_dev_para_eval_custom:
                            
                            print(f"\n--- Evaluación Detallada de Corrección de '{nombre_clase_corr_dev_config}' (Método Personalizado) ---")
                            
                            all_true_corr_dev_custom_eval = []
                            all_imputed_corr_dev_custom_eval = []
                            corr_dev_custom_metrics_eval = []

                            # Prepare evaluation DataFrames
                            df_original_ref_eval_corr_dev = df_original_eval_actual_ref.copy()
                            if df_original_ref_eval_corr_dev.index.has_duplicates:
                                df_original_ref_eval_corr_dev = df_original_ref_eval_corr_dev[~df_original_ref_eval_corr_dev.index.duplicated(keep='first')]
                            
                            df_corr_dev_corregido_eval_custom = df_corr_dev_corregido_custom.copy() 
                            if df_corr_dev_corregido_eval_custom.index.has_duplicates:
                                df_corr_dev_corregido_eval_custom = df_corr_dev_corregido_eval_custom[~df_corr_dev_corregido_eval_custom.index.duplicated(keep='first')]

                            eval_data_by_col_corr_dev = defaultdict(lambda: {'true': [], 'imputed': [], 'indices': []})

                            for col_eval_cd, idx_eval_cd in indices_corr_dev_para_eval_custom:
                                if not (idx_eval_cd in df_original_ref_eval_corr_dev.index and \
                                        idx_eval_cd in df_corr_dev_corregido_eval_custom.index and \
                                        col_eval_cd in df_original_ref_eval_corr_dev.columns and \
                                        col_eval_cd in df_corr_dev_corregido_eval_custom.columns):
                                    continue
                                
                                true_val_cd = df_original_ref_eval_corr_dev.loc[idx_eval_cd, col_eval_cd]
                                imputed_val_cd = df_corr_dev_corregido_eval_custom.loc[idx_eval_cd, col_eval_cd]

                                if pd.notna(true_val_cd) and pd.notna(imputed_val_cd): 
                                    eval_data_by_col_corr_dev[col_eval_cd]['true'].append(true_val_cd)
                                    eval_data_by_col_corr_dev[col_eval_cd]['imputed'].append(imputed_val_cd)
                                    eval_data_by_col_corr_dev[col_eval_cd]['indices'].append(idx_eval_cd)
                            
                            for col_eval_metric_cd, data_cd in eval_data_by_col_corr_dev.items():
                                if not data_cd['true']: continue
                                true_vals_series_cd = pd.Series(data_cd['true'])
                                imputed_vals_series_cd = pd.Series(data_cd['imputed'])
                                
                                all_true_corr_dev_custom_eval.extend(true_vals_series_cd.tolist())
                                all_imputed_corr_dev_custom_eval.extend(imputed_vals_series_cd.tolist())
                                
                                rmse_col_cd = np.sqrt(mean_squared_error(true_vals_series_cd, imputed_vals_series_cd))
                                mae_col_cd = mean_absolute_error(true_vals_series_cd, imputed_vals_series_cd)
                                
                                corr_dev_custom_metrics_eval.append({
                                    'Columna': col_eval_metric_cd,
                                    'Num_Puntos_Corregidos': len(true_vals_series_cd),
                                    'RMSE_Correccion_CorrDev': rmse_col_cd,
                                    'MAE_Correccion_CorrDev': mae_col_cd,
                                    'Media_Real_Original': true_vals_series_cd.mean(),
                                    'Media_Imputada_CorrDev': imputed_vals_series_cd.mean()
                                })

                            if corr_dev_custom_metrics_eval:
                                df_corr_dev_custom_eval_metrics = pd.DataFrame(corr_dev_custom_metrics_eval)
                                print(f"\nMétricas de Calidad de Corrección de '{nombre_clase_corr_dev_config}' (Método Personalizado) - Resumen por Columna:")
                                print(df_corr_dev_custom_eval_metrics.to_string())

                                if all_true_corr_dev_custom_eval and all_imputed_corr_dev_custom_eval:
                                    global_rmse_corr_dev_custom = np.sqrt(mean_squared_error(all_true_corr_dev_custom_eval, all_imputed_corr_dev_custom_eval))
                                    global_mae_corr_dev_custom = mean_absolute_error(all_true_corr_dev_custom_eval, all_imputed_corr_dev_custom_eval)
                                    print(f"\nRMSE Global (Corrección {nombre_clase_corr_dev_config} Custom): {global_rmse_corr_dev_custom:.3f} (sobre {len(all_true_corr_dev_custom_eval)} puntos)")
                                    print(f"MAE Global (Corrección {nombre_clase_corr_dev_config} Custom): {global_mae_corr_dev_custom:.3f} (sobre {len(all_true_corr_dev_custom_eval)} puntos)")
                                    
                                    plt.figure(figsize=(8, 8))
                                    sample_size_corr_dev = min(len(all_true_corr_dev_custom_eval), 2000)
                                    if sample_size_corr_dev > 0:
                                        indices_smp_corr_dev = np.random.choice(len(all_true_corr_dev_custom_eval), size=sample_size_corr_dev, replace=False)
                                        true_smp_cd = np.array(all_true_corr_dev_custom_eval)[indices_smp_corr_dev]
                                        imputed_smp_cd = np.array(all_imputed_corr_dev_custom_eval)[indices_smp_corr_dev]
                                        
                                        plt.scatter(true_smp_cd, imputed_smp_cd, alpha=0.5, s=10, label=f"Corrected Points", c='orange')
                                        min_val_plot_cd = np.nanmin(np.concatenate([true_smp_cd, imputed_smp_cd]))
                                        max_val_plot_cd = np.nanmax(np.concatenate([true_smp_cd, imputed_smp_cd]))
                                        if pd.notna(min_val_plot_cd) and pd.notna(max_val_plot_cd) and min_val_plot_cd < max_val_plot_cd:
                                            plt.plot([min_val_plot_cd, max_val_plot_cd], [min_val_plot_cd, max_val_plot_cd], 'r--', lw=2, label='Ideal')
                                        plt.xlabel("True Values")
                                        plt.ylabel(f"Corrected Values")
                                        plt.legend()
                                        plt.grid(True)
                                        plt.tight_layout()
                                        plt.show()
                                else:
                                    print("\nNo hay suficientes datos globales para calcular RMSE/MAE para la corrección de Desviación de Correlación.")
                            else:
                                print(f"\nNo se generaron métricas para la corrección de '{nombre_clase_corr_dev_config}'.")
                        else:
                            print(f"\nNo se pudo realizar la evaluación de corrección de '{nombre_clase_corr_dev_config}' (df_original_eval_actual no disponible o no se identificaron puntos para evaluar).")

                    print("\n===============================================================================")
                    print(f"FIN DE LA FASE DE CORRECCIÓN DE '{nombre_clase_corr_dev_config}' (MÉTODO PERSONALIZADO)")
                    print("===============================================================================")
                    print(f"\nEl DataFrame con las correcciones ('{nombre_clase_corr_dev_config}' con método custom) está en: df_corr_dev_corregido_custom")
                
                else: 
                    print(f"Error: 'columnas_objetivo_existentes' no está disponible. No se puede proceder con la corrección de '{nombre_clase_corr_dev_config}'.")
            else: 
                print(f"Error: 'std_devs_originales' no está disponible. No se puede proceder con la corrección de '{nombre_clase_corr_dev_config}'.")
        else:
            print(f"Error: El DataFrame 'df_oor_corregido' no está disponible. Asegúrate de que los pasos anteriores se hayan ejecutado correctamente para poder iniciar la corrección de '{nombre_clase_corr_dev_config}'.")

## 10. Corrección de Anomalías Contextuales

In [ ]:
# --- INICIO DEL PROCESO DE CORRECCIÓN DE "ANOMALÍAS CONTEXTUALES" (MÉTODO PERSONALIZADO) ---

nombre_clase_contextual_config = "Contextual"

# Factor para el umbral de diferencia entre anterior y siguiente para esta corrección
# v3 — M4
factor_umbral_diferencia_contextual = FACTOR_UMBRAL_CORRECCION

# Verificar que las variables necesarias existen
# El DataFrame de entrada es df_corr_dev_corregido_custom (resultado de la corrección anterior)
if 'df_corr_dev_corregido_custom' in locals() or 'df_corr_dev_corregido_custom' in globals():
    
    df_temp_ctx = locals().get('df_corr_dev_corregido_custom', globals().get('df_corr_dev_corregido_custom'))
    if 'pred_tipo_anomalia' in df_temp_ctx.columns:
        serie_predicciones_hybrid = df_temp_ctx['pred_tipo_anomalia']
        print(f"\n✅ Columna 'pred_tipo_anomalia' encontrada. Se filtrarán solo los predecidos como '{nombre_clase_contextual_config}'.")
    elif 'serie_predicciones_hybrid' not in locals() and 'serie_predicciones_hybrid' not in globals():
        # Intentamos reconstruir a partir del modelo
        y_pred = locals().get('y_pred_model2_actual', globals().get('y_pred_model2_actual', locals().get('y_pred_model2', globals().get('y_pred_model2'))))
        le_model = locals().get('label_encoder_model2_actual', globals().get('label_encoder_model2_actual', locals().get('label_encoder_model2', globals().get('label_encoder_model2'))))
        X_test = locals().get('X_model2_test_actual', globals().get('X_model2_test_actual', locals().get('X_model2_test', globals().get('X_model2_test'))))
        
        if y_pred is not None and le_model is not None and X_test is not None:
            try:
                serie_pred_temp = pd.Series(le_model.inverse_transform(y_pred), index=X_test.index)
                serie_predicciones_hybrid = pd.Series(dtype='object', index=df_temp_ctx.index)
                common_idx = df_temp_ctx.index.intersection(serie_pred_temp.index)
                serie_predicciones_hybrid.loc[common_idx] = serie_pred_temp.loc[common_idx]
                print(f"\n✅ Se usaron las variables del modelo para generar las predicciones de '{nombre_clase_contextual_config}'.")
            except Exception:
                serie_predicciones_hybrid = pd.Series(dtype='object')
        else:
            serie_predicciones_hybrid = pd.Series(dtype='object')
    else:
        serie_predicciones_hybrid = locals().get('serie_predicciones_hybrid', globals().get('serie_predicciones_hybrid'))

        if 'std_devs_originales' in locals() or 'std_devs_originales' in globals():
            if 'columnas_objetivo_existentes' in locals() or 'columnas_objetivo_existentes' in globals():

                    print("\n===============================================================================")
                    print(f"INICIANDO FASE DE CORRECCIÓN DE '{nombre_clase_contextual_config}' CON MÉTODO PERSONALIZADO")
                    print(f"Factor umbral para diferencia (x std_dev): {factor_umbral_diferencia_contextual}")
                    print("===============================================================================")

                    df_contextual_corregido_custom = df_corr_dev_corregido_custom.copy()
                    
                    columnas_para_corregir_contextual_custom = [
                        col for col in columnas_objetivo_existentes if col in df_contextual_corregido_custom.columns
                    ]
                    
                    if not columnas_para_corregir_contextual_custom:
                        print(f"Advertencia: No hay columnas objetivo válidas para la corrección de '{nombre_clase_contextual_config}' personalizada.")
                    else:
                        print(f"Columnas objetivo para corrección de '{nombre_clase_contextual_config}': {columnas_para_corregir_contextual_custom}")

                        puntos_contextual_corregidos_count = {}
                        indices_contextual_para_eval_custom = [] 

                        print(f"\n--- Identificando y Corrigiendo '{nombre_clase_contextual_config}' con lógica personalizada ---")
                        
                        lista_puntos_contextual_identificados = []
                        print(f"Fase 1: Identificando todos los puntos con '{nombre_clase_contextual_config}'...")
                        for col_idx_check in columnas_para_corregir_contextual_custom:
                            if col_idx_check not in df_contextual_corregido_custom.columns:
                                continue
                            
                            temp_serie_pred_alineada_contextual = pd.Series(dtype=object)
                            current_df_index_contextual = df_contextual_corregido_custom.index
                            predictions_index_contextual = serie_predicciones_hybrid.index

                            if not isinstance(current_df_index_contextual, pd.Index) or not isinstance(predictions_index_contextual, pd.Index):
                                print(f"Advertencia (Alineación Contextual Custom): Índices no válidos para '{col_idx_check}'.")
                                continue

                            if current_df_index_contextual.equals(predictions_index_contextual):
                                temp_serie_pred_alineada_contextual = serie_predicciones_hybrid
                            else:
                                common_idx_pred_contextual = current_df_index_contextual.intersection(predictions_index_contextual)
                                if not common_idx_pred_contextual.empty:
                                    temp_serie_pred_alineada_contextual = serie_predicciones_hybrid.loc[common_idx_pred_contextual].reindex(current_df_index_contextual)
                                else:
                                    continue
                            
                            if temp_serie_pred_alineada_contextual.empty or temp_serie_pred_alineada_contextual.isnull().all():
                                continue

                            indices_contextual_col = temp_serie_pred_alineada_contextual[temp_serie_pred_alineada_contextual == nombre_clase_contextual_config].index
                            for idx_contextual_item in indices_contextual_col:
                                if idx_contextual_item in df_contextual_corregido_custom.index: 
                                    lista_puntos_contextual_identificados.append((col_idx_check, idx_contextual_item))
                                    indices_contextual_para_eval_custom.append((col_idx_check, idx_contextual_item)) 
                        
                        print(f"Fase 1: {len(lista_puntos_contextual_identificados)} puntos con '{nombre_clase_contextual_config}' identificados en total.")
                        
                        df_source_for_neighbors_contextual = df_corr_dev_corregido_custom.copy()
                        if df_source_for_neighbors_contextual.index.has_duplicates:
                            print(f"Advertencia (Contextual Custom - Preparación Vecinos): DataFrame fuente ('df_corr_dev_corregido_custom') para vecinos tiene duplicados. Se tomará la primera aparición.")
                            df_source_for_neighbors_contextual = df_source_for_neighbors_contextual[~df_source_for_neighbors_contextual.index.duplicated(keep='first')]
                        
                        print("Fase 2: Aplicando corrección a los puntos identificados...")
                        for col_actual, idx_contextual in lista_puntos_contextual_identificados:
                            puntos_contextual_corregidos_count[col_actual] = puntos_contextual_corregidos_count.get(col_actual, 0) + 1
                            
                            i_pos = -1 
                            try:
                                loc_result = df_source_for_neighbors_contextual.index.get_loc(idx_contextual)
                                if isinstance(loc_result, int): i_pos = loc_result
                                elif isinstance(loc_result, slice):
                                    i_pos = loc_result.start
                                    if i_pos is None: i_pos = -1
                                elif isinstance(loc_result, np.ndarray): 
                                    if loc_result.dtype == bool: positions = np.where(loc_result)[0]
                                    else: positions = loc_result
                                    if len(positions) > 0: i_pos = positions[0] 
                                    else: i_pos = -1
                                if i_pos == -1:
                                    df_contextual_corregido_custom.loc[idx_contextual, col_actual] = np.nan
                                    continue
                            except KeyError:
                                df_contextual_corregido_custom.loc[idx_contextual, col_actual] = np.nan
                                continue
                            except Exception as e_getloc:
                                print(f"Error inesperado en get_loc (Contextual) para índice {idx_contextual}, columna {col_actual}: {e_getloc}. Saltando.")
                                df_contextual_corregido_custom.loc[idx_contextual, col_actual] = np.nan
                                continue

                            val_prev_original, val_next_original = np.nan, np.nan
                            prev_val_exists, next_val_exists = False, False

                            if 0 <= i_pos < len(df_source_for_neighbors_contextual.index):
                                if i_pos > 0:
                                    idx_prev_ts = df_source_for_neighbors_contextual.index[i_pos - 1]
                                    val_prev_original = df_source_for_neighbors_contextual.loc[idx_prev_ts, col_actual] 
                                    if pd.notna(val_prev_original): prev_val_exists = True
                                if i_pos < len(df_source_for_neighbors_contextual.index) - 1:
                                    idx_next_ts = df_source_for_neighbors_contextual.index[i_pos + 1]
                                    val_next_original = df_source_for_neighbors_contextual.loc[idx_next_ts, col_actual]
                                    if pd.notna(val_next_original): next_val_exists = True
                            else:
                                df_contextual_corregido_custom.loc[idx_contextual, col_actual] = np.nan
                                continue

                            corrected_value = np.nan 
                            if prev_val_exists and next_val_exists:
                                diff = abs(val_next_original - val_prev_original)
                                std_dev_col = std_devs_originales.get(col_actual, 0) 
                                threshold = factor_umbral_diferencia_contextual * std_dev_col if std_dev_col > 0 else float('inf')
                                if diff > threshold : corrected_value = val_prev_original 
                                else: corrected_value = (val_prev_original + val_next_original) / 2
                            elif prev_val_exists: corrected_value = val_prev_original
                            elif next_val_exists: corrected_value = val_next_original
                            
                            df_contextual_corregido_custom.loc[idx_contextual, col_actual] = corrected_value

                        for col_rep, count_rep in puntos_contextual_corregidos_count.items():
                            if count_rep > 0:
                                print(f"   En columna '{col_rep}': {count_rep} puntos '{nombre_clase_contextual_config}' procesados con método personalizado.")
                        
                        # --- Evaluación de la Corrección de Anomalías Contextuales Personalizada ---
                        df_original_eval_actual_ref = locals().get('df_original_eval_actual', globals().get('df_original_eval_actual'))
                        if df_original_eval_actual_ref is None:
                            df_original_eval_actual_ref = locals().get('df_original', globals().get('df_original'))

                        if df_original_eval_actual_ref is not None and indices_contextual_para_eval_custom:
                            
                            print(f"\n--- Evaluación Detallada de Corrección de '{nombre_clase_contextual_config}' (Método Personalizado) ---")
                            
                            all_true_contextual_custom_eval = []
                            all_imputed_contextual_custom_eval = []
                            contextual_custom_metrics_eval = []

                            df_original_ref_eval_contextual = df_original_eval_actual_ref.copy()
                            if df_original_ref_eval_contextual.index.has_duplicates:
                                df_original_ref_eval_contextual = df_original_ref_eval_contextual[~df_original_ref_eval_contextual.index.duplicated(keep='first')]
                            
                            df_contextual_corregido_eval_custom = df_contextual_corregido_custom.copy() 
                            if df_contextual_corregido_eval_custom.index.has_duplicates:
                                df_contextual_corregido_eval_custom = df_contextual_corregido_eval_custom[~df_contextual_corregido_eval_custom.index.duplicated(keep='first')]

                            eval_data_by_col_contextual = defaultdict(lambda: {'true': [], 'imputed': [], 'indices': []})

                            for col_eval_ctx, idx_eval_ctx in indices_contextual_para_eval_custom:
                                if not (idx_eval_ctx in df_original_ref_eval_contextual.index and \
                                        idx_eval_ctx in df_contextual_corregido_eval_custom.index and \
                                        col_eval_ctx in df_original_ref_eval_contextual.columns and \
                                        col_eval_ctx in df_contextual_corregido_eval_custom.columns):
                                    continue
                                true_val_ctx = df_original_ref_eval_contextual.loc[idx_eval_ctx, col_eval_ctx]
                                imputed_val_ctx = df_contextual_corregido_eval_custom.loc[idx_eval_ctx, col_eval_ctx]
                                if pd.notna(true_val_ctx) and pd.notna(imputed_val_ctx): 
                                    eval_data_by_col_contextual[col_eval_ctx]['true'].append(true_val_ctx)
                                    eval_data_by_col_contextual[col_eval_ctx]['imputed'].append(imputed_val_ctx)
                                    eval_data_by_col_contextual[col_eval_ctx]['indices'].append(idx_eval_ctx)
                            
                            for col_eval_metric_ctx, data_ctx in eval_data_by_col_contextual.items():
                                if not data_ctx['true']: continue
                                true_vals_series_ctx = pd.Series(data_ctx['true'])
                                imputed_vals_series_ctx = pd.Series(data_ctx['imputed'])
                                all_true_contextual_custom_eval.extend(true_vals_series_ctx.tolist())
                                all_imputed_contextual_custom_eval.extend(imputed_vals_series_ctx.tolist())
                                rmse_col_ctx = np.sqrt(mean_squared_error(true_vals_series_ctx, imputed_vals_series_ctx))
                                mae_col_ctx = mean_absolute_error(true_vals_series_ctx, imputed_vals_series_ctx)
                                contextual_custom_metrics_eval.append({
                                    'Columna': col_eval_metric_ctx,
                                    'Num_Puntos_Corregidos': len(true_vals_series_ctx),
                                    'RMSE_Correccion_Contextual': rmse_col_ctx,
                                    'MAE_Correccion_Contextual': mae_col_ctx,
                                    'Media_Real_Original': true_vals_series_ctx.mean(),
                                    'Media_Imputada_Contextual': imputed_vals_series_ctx.mean()
                                })

                            if contextual_custom_metrics_eval:
                                df_contextual_custom_eval_metrics = pd.DataFrame(contextual_custom_metrics_eval)
                                print(f"\nMétricas de Calidad de Corrección de '{nombre_clase_contextual_config}' (Método Personalizado) - Resumen por Columna:")
                                print(df_contextual_custom_eval_metrics.to_string())

                                if all_true_contextual_custom_eval and all_imputed_contextual_custom_eval:
                                    global_rmse_contextual_custom = np.sqrt(mean_squared_error(all_true_contextual_custom_eval, all_imputed_contextual_custom_eval))
                                    global_mae_contextual_custom = mean_absolute_error(all_true_contextual_custom_eval, all_imputed_contextual_custom_eval)
                                    print(f"\nRMSE Global (Corrección {nombre_clase_contextual_config} Custom): {global_rmse_contextual_custom:.3f} (sobre {len(all_true_contextual_custom_eval)} puntos)")
                                    print(f"MAE Global (Corrección {nombre_clase_contextual_config} Custom): {global_mae_contextual_custom:.3f} (sobre {len(all_true_contextual_custom_eval)} puntos)")
                                    
                                    plt.figure(figsize=(8, 8))
                                    sample_size_contextual = min(len(all_true_contextual_custom_eval), 2000)
                                    if sample_size_contextual > 0:
                                        indices_smp_contextual = np.random.choice(len(all_true_contextual_custom_eval), size=sample_size_contextual, replace=False)
                                        true_smp_ctx = np.array(all_true_contextual_custom_eval)[indices_smp_contextual]
                                        imputed_smp_ctx = np.array(all_imputed_contextual_custom_eval)[indices_smp_contextual]
                                        plt.scatter(true_smp_ctx, imputed_smp_ctx, alpha=0.5, s=10, label=f"Corrected Points", c='purple') # Changed color for distinction
                                        min_val_plot_ctx = np.nanmin(np.concatenate([true_smp_ctx, imputed_smp_ctx]))
                                        max_val_plot_ctx = np.nanmax(np.concatenate([true_smp_ctx, imputed_smp_ctx]))
                                        if pd.notna(min_val_plot_ctx) and pd.notna(max_val_plot_ctx) and min_val_plot_ctx < max_val_plot_ctx:
                                            plt.plot([min_val_plot_ctx, max_val_plot_ctx], [min_val_plot_ctx, max_val_plot_ctx], 'r--', lw=2, label='Ideal')
                                        plt.xlabel("True Values")
                                        plt.ylabel("Corrected Values")
                                        plt.legend()
                                        plt.grid(True)
                                        plt.tight_layout()
                                        plt.show()
                                else:
                                    print("\nNo hay suficientes datos globales para calcular RMSE/MAE para la corrección de Anomalías Contextuales.")
                            else:
                                print(f"\nNo se generaron métricas para la corrección de '{nombre_clase_contextual_config}'.")
                        else:
                            print(f"\nNo se pudo realizar la evaluación de corrección de '{nombre_clase_contextual_config}' (df_original_eval_actual no disponible o no se identificaron puntos para evaluar).")

                    print("\n===============================================================================")
                    print(f"FIN DE LA FASE DE CORRECCIÓN DE '{nombre_clase_contextual_config}' (MÉTODO PERSONALIZADO)")
                    print("===============================================================================")
                    print(f"\nEl DataFrame con las correcciones ('{nombre_clase_contextual_config}' con método custom) está en: df_contextual_corregido_custom")
            else: 
                print(f"Error: 'columnas_objetivo_existentes' no está disponible. No se puede proceder con la corrección de '{nombre_clase_contextual_config}'.")
        else: 
            print(f"Error: 'std_devs_originales' no está disponible. No se puede proceder con la corrección de '{nombre_clase_contextual_config}'.")
else:
    print(f"Error: El DataFrame 'df_corr_dev_corregido_custom' no está disponible. Asegúrate de que los pasos anteriores se hayan ejecutado correctamente para poder iniciar la corrección de '{nombre_clase_contextual_config}'.")

## 11. Guardar resultado corregido

In [ ]:
# El dataframe final con todas las correcciones aplicadas es df_contextual_corregido_custom
# (o el último dataframe de la cadena de correcciones)
df_corregido_final = df_contextual_corregido_custom.copy() if 'df_contextual_corregido_custom' in locals() else df_stuck_hybrid_corrected.copy()
# Restaurar Fecha como columna si quedó como DatetimeIndex durante la corrección
if isinstance(df_corregido_final.index, pd.DatetimeIndex) and df_corregido_final.index.name == 'Fecha':
    df_corregido_final = df_corregido_final.reset_index()
df_corregido_final.to_parquet(PARQUET_06, index=False)

# También guardamos el imputer de datos faltantes para usarlo en inferencia real
joblib.dump(iterative_imputer_faltantes, IMPUTER_FALT_PATH)

print(f"Resultado corregido guardado: {PARQUET_06}")
print(f"Shape: {df_corregido_final.shape}")
